# V9 — DINOv3 ViT-7B frozen: audited drop vs relabel

This notebook is a train-only, cache-only experiment. It compares deleting obvious label-noise rows against relabeling the same human-confirmed rows while keeping the official validation folds completely unchanged.

The DINOv3 ViT-7B LVD backbone remains frozen and is never loaded or forwarded in this notebook.


## 2. Locked project direction

- Backbone: `facebook/dinov3-vit7b16-pretrain-lvd1689m`, immutable revision, 100% frozen.
- Representation: V7 final-block `concat(CLS, mean true patch tokens)` cache.
- Control: exact V7 `regularized_mlp_final`.
- V9 treatments are applied **only to the training partition of each fold**.
- Validation rows and validation labels always remain the original official labels.
- No test access, no pseudo-labeling, no submission creation, no backbone loading.


## 3. V7/V8 evidence audit

Repository review verified full five-fold V7 selected and baseline OOF predictions. V7 selected Macro F1 is 0.990136 across 26,527 rows with 287 errors, 241 Recyclable↔Organic; Electronic F1 is 0.994205. The two V7 heads make the same wrong alternative on 230 rows. V8 reproduced fold 0 exactly, but its 0.75 specialist blend fell to 0.989322 with 65 errors and full replacement fell to 0.989472 with 64 errors, versus 0.989918 and 61 errors for V7. V8 correctly stopped. The active papers support conservative feature-space experimentation generally, but none supplies project-specific evidence strong enough to override these direct OOF results.

The reviewed V7 gallery contains ambiguous packaging/content cases as well as suspicious examples. That supports bounded human review, not automatic semantic deletion.

## 4. Competing hypotheses and selected V9 family

V9 directly compares two treatments for the same bounded human-reviewed rows:

- **C1 drop-only:** drop the two objective duplicate conflicts and drop the ten obvious semantic mislabels.
- **C2 relabel:** drop only the two objective duplicate conflicts, but relabel the ten obvious semantic mislabels to their reviewed classes.
- **C3 blend:** fixed 50/50 probability blend of C0 and C2.

This comparison is pre-registered before fold-0 scoring. Ambiguous rows remain unchanged.


## 5. Deterministic configuration and `RUN_MODE`

Modes:

- `AUDIT`: regenerate the review bundle from official raw train.
- `SCREEN`: consume the audit bundle, apply the locked 10 relabel + 2 objective-drop decisions, reproduce C0, then compare C1/C2/C3 on fold 0.
- `FULL_CONFIRMATION`: requires a passed hash-matched SCREEN bundle and runs folds 1–4 without retuning.


In [1]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from torch.utils.data import DataLoader, Dataset

STARTED_AT = time.perf_counter()
DEFAULT_RUN_MODE = 'SCREEN'
ALLOWED_RUN_MODES = ('AUDIT', 'SCREEN', 'FULL_CONFIRMATION')
RUN_MODE = os.environ.get('V9_RUN_MODE', DEFAULT_RUN_MODE).strip().upper()
if RUN_MODE not in ALLOWED_RUN_MODES:
    raise RuntimeError('V9_RUN_MODE must be one of {}.'.format(ALLOWED_RUN_MODES))

EXPERIMENT_NAME = 'v9_dinov3_vit7b_frozen_drop_vs_relabel'
V9_SCHEMA_VERSION = 2
V9_REVISION = 'v9_crossfitted_drop_vs_relabel_v2'
V7_DIRECTORY = 'v7_dinov3_vit7b_frozen_raw'
V8_DIRECTORY = 'v8_dinov3_vit7b_frozen_cache_upgrade'
MODEL_ID = 'facebook/dinov3-vit7b16-pretrain-lvd1689m'
MODEL_REVISION = 'b80367753773648a6793235ab9c65cdbb029506f'
CACHE_SIGNATURE_EXPECTED = 'c722dc39a8cba286d3aba34d88f3e8f87071353bc6585e755873659de8560ddc'
FOLD_SIGNATURE_EXPECTED = '0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c'
IMAGE_IDS_SHA256_EXPECTED = '210f8f14797be49bad962e81f1da660a257ec28ee2136b54a7516950a66cb2e8'
LABELS_SHA256_EXPECTED = '719d36e9918b1fd09711c06d1b259fca08ca6eb4d973de1388d5e3ddecf22d35'
RAW_DATA_SHA256_EXPECTED = '73bd0d3c17e75e7c0f10e4324ffa9ef81f6224cd2b3ba0e53e9f2aea7fde3212'
V7_CACHE_REVISION = 'v7_official_final_norm_before_pooling_v2'
V8_REVISION = 'v8_ro_hierarchical_specialist_final_cache_v1'
EXPECTED_ROWS = 26527
EXPECTED_CLASS_COUNTS = {0: 9999, 1: 3961, 2: 12567}
CLASS_NAMES = ('Recyclable', 'Electronic', 'Organic')
CLASS_MAPPING = {0: 'Recyclable', 1: 'Electronic', 2: 'Organic'}
N_CLASSES = 3
N_FOLDS = 5
DEVELOPMENT_FOLD = 0
CONFIRMATION_FOLDS = (1, 2, 3, 4)
SEED = 42
HIDDEN_SIZE = 4096
LAST_BLOCKS = (36, 37, 38, 39)
LAST_BLOCK_NAMES = ('layer.36', 'layer.37', 'layer.38', 'layer.39')
FEATURE_SHAPE = (EXPECTED_ROWS, 4, 2, HIDDEN_SIZE)
CONTROL_NAME = 'regularized_mlp_final'
CANDIDATE_NAMES = ('C1', 'C2', 'C3')
HEAD_HIDDEN = 512
DROPOUT = 0.40
LABEL_SMOOTHING = 0.05
MAX_EPOCHS = 15
PATIENCE = 5
LEARNING_RATE = 8e-4
WEIGHT_DECAY = 1e-4
TRAIN_BATCH_SIZE = 256
PREDICT_BATCH_SIZE = 512
GRAD_CLIP = 1.0
CONTROL_TOLERANCE = 0.0002
BLEND_WEIGHT_C0 = 0.50
BLEND_WEIGHT_C2 = 0.50
EXPECTED_V7_FOLD0_MACRO_F1 = 0.9899180117906018
EXPECTED_V7_FOLD0_ERRORS = 61
EXPECTED_V7_FOLD0_R2O = 24
EXPECTED_V7_FOLD0_O2R = 30
EXPECTED_V7_FOLD0_E_F1 = 0.9955891619407687
EXPECTED_V7_LAST4_MACRO_F1 = 0.9880727960809867
EXPECTED_V7_CONFIRMATION_MACRO_F1 = 0.990190599402788
EXPECTED_V7_CONFIRMATION_TOTAL_ERRORS = 226
EXPECTED_V7_CONFIRMATION_RO_ERRORS = 187
EXPECTED_V7_CONFIRMATION_E_F1 = 0.9938592347661785
EXPECTED_V7_FIVEFOLD_MACRO_F1 = 0.9901360067264733
EXPECTED_V7_FIVEFOLD_ERRORS = 287
EXPECTED_V7_FIVEFOLD_RO_ERRORS = 241
EXPECTED_V7_FIVEFOLD_E_F1 = 0.9942050894431846
EXPECTED_V7_CONSENSUS_WRONG = 230
EXPECTED_PREFLIGHT_SIGNATURE = '961831943107801b06846ea7dfa69877601e0d2e0defd6e71c3a14c97ffa64cf'
EXPECTED_V8_BLEND_MACRO_F1 = 0.9893223868666294
EXPECTED_V8_FULL_MACRO_F1 = 0.9894724316433141
SEMANTIC_QUEUE = {'selected_top_min': 0.90, 'selected_original_max': 0.05, 'selected_margin_min': 0.80, 'baseline_top_min': 0.99, 'baseline_original_max': 0.005}
BLANK_STD_MAX = 1.0
BLANK_RANGE_MAX = 4
AUDIT_THUMBNAIL_SIDE = 256
CONTACT_SHEET_IMAGES = 20
MAX_CONTACT_SHEETS = 12
MAX_TOTAL_DROPS = 132
MAX_TOTAL_RELABELS = 50
MAX_DROP_FRACTION_PER_CLASS = 0.01
MAX_CLASS_PROPORTION_SHIFT = 0.003
ALLOWED_ACTIONS = ('KEEP', 'DROP_OBVIOUS_MISLABEL', 'DROP_CORRUPT_OR_UNUSABLE', 'RELABEL_OBVIOUS_MISLABEL', 'AMBIGUOUS_KEEP')
TIER_A = 'TIER_A_AUTOMATIC_SAFE_DROP'
TIER_B = 'TIER_B_MANUAL_OBVIOUS_MISLABEL'
TIER_C = 'TIER_C_AMBIGUOUS_KEEP'
TIER_KEEP = 'KEEP'
FOLD0_GATES = {'minimum_macro_f1': 0.9903, 'minimum_gain': 0.0003, 'maximum_total_errors': 59, 'minimum_ro_error_reduction': 3, 'maximum_electronic_f1_degradation': 0.0005, 'minimum_corrected': 5, 'minimum_net_corrections': 2}
FULL_GATES = {'minimum_macro_f1': 0.9905, 'minimum_gain': 0.0003, 'minimum_folds_beating_control': 3, 'maximum_total_errors': 279, 'maximum_ro_errors_exclusive': EXPECTED_V7_FIVEFOLD_RO_ERRORS, 'maximum_electronic_f1_degradation': 0.0005, 'maximum_positive_net_fold_share': 0.75, 'minimum_corrected_nonreviewed': 3}
TRAIN_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
AUDIT_CONTRACT_COLUMNS = ('sample_index', 'relative_path', 'fold', 'original_label', 'oof_predicted_label', 'oof_original_probability', 'oof_top_probability', 'oof_margin', 'selected_baseline_consensus', 'candidate_tier', 'review_action', 'review_reason', 'reviewer', 'review_timestamp', 'evidence_source')
REVIEW_REQUIRED_COLUMNS = ('sample_index', 'relative_path', 'fold', 'original_label', 'reviewed_label', 'oof_predicted_label', 'oof_original_probability', 'oof_top_probability', 'oof_margin', 'selected_baseline_consensus', 'candidate_tier', 'review_action', 'review_reason', 'reviewer', 'review_timestamp', 'evidence_source')
LOCKED_REVIEW_TIMESTAMP = '2026-07-28T00:00:00+07:00'
LOCKED_REVIEWER = 'Riantama Putra'
LOCKED_RELABEL_DECISIONS = {
    '0_Recyclable/R_3825.jpg': {'original_label': 0, 'reviewed_label': 1, 'reason': 'Keyboard is clearly an electronic device.'},
    '0_Recyclable/R_8295.jpg': {'original_label': 0, 'reviewed_label': 1, 'reason': 'Laptop is clearly an electronic device.'},
    '2_Organic/O_7701.jpg': {'original_label': 2, 'reviewed_label': 1, 'reason': 'Microwave oven is clearly an electronic device.'},
    '2_Organic/O_7776.jpg': {'original_label': 2, 'reviewed_label': 1, 'reason': 'Microwave oven is clearly an electronic device.'},
    '0_Recyclable/R_9775.jpg': {'original_label': 0, 'reviewed_label': 2, 'reason': 'Image clearly shows food.'},
    '0_Recyclable/R_8580.jpg': {'original_label': 0, 'reviewed_label': 2, 'reason': 'Image clearly shows noodles/food.'},
    '2_Organic/O_1949.jpg': {'original_label': 2, 'reviewed_label': 0, 'reason': 'Image clearly shows manufactured plastic pitchers.'},
    '2_Organic/O_9175.jpg': {'original_label': 2, 'reviewed_label': 0, 'reason': 'Image clearly shows a manufactured notebook.'},
    '2_Organic/O_1807.jpg': {'original_label': 2, 'reviewed_label': 0, 'reason': 'Image clearly shows a manufactured plastic bottle.'},
    '2_Organic/O_1978.jpg': {'original_label': 2, 'reviewed_label': 0, 'reason': 'Image clearly shows a paper envelope.'},
}
LOCKED_OBJECTIVE_DROP_PATHS = ('0_Recyclable/R_799.jpg', '2_Organic/O_8873.jpg')

KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError('V9 requires /kaggle/working in a Kaggle runtime.')
ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
HEADS_DIR = ARTIFACT_DIR / 'heads'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
HEADS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def json_default(value):
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    if torch.is_tensor(value):
        return value.detach().cpu().tolist()
    raise TypeError(type(value).__name__)

def write_json(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, sort_keys=True, allow_nan=False, default=json_default)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

def write_text(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        handle.write(value)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def read_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)

def sha256_json(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, default=json_default)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def candidate_contract_records(frame):
    immutable = [column for column in AUDIT_CONTRACT_COLUMNS if column not in {'review_action', 'review_reason', 'reviewer', 'review_timestamp'}]
    normalized = frame[immutable].copy().sort_values('sample_index').reset_index(drop=True)
    for column in ('sample_index', 'fold', 'original_label', 'oof_predicted_label'):
        normalized[column] = pd.to_numeric(normalized[column], errors='raise').astype(np.int64)
    for column in ('oof_original_probability', 'oof_top_probability', 'oof_margin'):
        normalized[column] = pd.to_numeric(normalized[column], errors='raise').round(12)
    consensus = normalized['selected_baseline_consensus'].astype(str).str.strip().str.lower().map({'true': True, 'false': False})
    require(consensus.notna().all(), 'Invalid selected/baseline consensus values.')
    normalized['selected_baseline_consensus'] = consensus.astype(bool)
    for column in ('relative_path', 'candidate_tier', 'evidence_source'):
        normalized[column] = normalized[column].astype(str)
    return normalized.to_dict(orient='records')

def candidate_contract_sha256(frame):
    return sha256_json(candidate_contract_records(frame))

def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('wb') as handle:
        torch.save(value, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def fail_closed(stage, message, context=None):
    report = {'schema_version': V9_SCHEMA_VERSION, 'revision': V9_REVISION, 'run_mode': RUN_MODE, 'pass': False, 'stage': stage, 'error': str(message), 'context': context or {}, 'official_test_accessed': False, 'submission_created': False}
    write_json(report, ARTIFACT_DIR / 'failure_report.json')
    raise RuntimeError('{}: {}'.format(stage, message))

def iter_strings(value):
    if isinstance(value, str):
        yield value
    elif isinstance(value, dict):
        for nested in value.values():
            yield from iter_strings(nested)
    elif isinstance(value, (list, tuple)):
        for nested in value:
            yield from iter_strings(nested)

def contains_forbidden_test_reference(value):
    for text in iter_strings(value):
        tokens = [token.lower() for token in text.replace(chr(92), '/').split('/') if token]
        if any(token in {'test', 'template.csv', 'submission.csv'} for token in tokens):
            return True
    return False

def normalize_probabilities(probabilities):
    probabilities = np.asarray(probabilities, dtype=np.float64).copy()
    require(probabilities.ndim == 2 and np.isfinite(probabilities).all() and (probabilities >= 0).all(), 'Invalid probabilities.')
    sums = probabilities.sum(axis=1, keepdims=True)
    require((sums > 0).all() and np.isfinite(sums).all(), 'Invalid probability row sums.')
    probabilities = probabilities.astype(np.float64)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    require(np.isfinite(probabilities).all() and np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-12, rtol=0.0), 'Probability normalization failed.')
    return probabilities

def probabilities_from_logits(logits):
    logits = np.asarray(logits, dtype=np.float64)
    require(logits.ndim == 2 and np.isfinite(logits).all(), 'Invalid logits.')
    shifted = logits - logits.max(axis=1, keepdims=True)
    return normalize_probabilities(np.exp(shifted))

def metric_bundle(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = normalize_probabilities(probabilities)
    require(probabilities.shape == (len(labels), N_CLASSES), 'Metric shape mismatch.')
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predictions, labels=list(range(N_CLASSES)))
    report = classification_report(labels, predictions, labels=list(range(N_CLASSES)), target_names=list(CLASS_NAMES), output_dict=True, zero_division=0)
    return {'rows': len(labels), 'macro_f1': float(f1_score(labels, predictions, labels=list(range(N_CLASSES)), average='macro')), 'accuracy': float(accuracy_score(labels, predictions)), 'multiclass_log_loss': float(log_loss(labels, probabilities, labels=list(range(N_CLASSES)))), 'classification_report': report, 'confusion_matrix': matrix.tolist(), 'recyclable_to_organic': int(matrix[0, 2]), 'organic_to_recyclable': int(matrix[2, 0]), 'total_errors': int((predictions != labels).sum())}

def provenance(manifest_hash=None):
    return {'schema_version': V9_SCHEMA_VERSION, 'revision': V9_REVISION, 'run_mode': RUN_MODE, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'cache_signature': CACHE_SIGNATURE_EXPECTED, 'fold_signature': FOLD_SIGNATURE_EXPECTED, 'cleaning_manifest_sha256': manifest_hash, 'seed': SEED, 'official_test_accessed': False, 'submission_created': False}

seed_everything(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_float32_matmul_precision('high')

COMPETITION_CLARIFICATIONS = {**provenance(), 'source': 'user-supplied official administrator participant-group clarification, corroborated where noted by Petunjuk Teknis Draft V1.26.0.2', 'guide_reference': 'Petunjuk_Teknis_BDC_Satria_Data_2026.md reviewed as repository text during static notebook creation', 'clarifications': [{'id': 1, 'text': 'Public pretrained DINOv2/DINOv3 is permitted when the final model is trained on official train.', 'guide_corroborated': True}, {'id': 2, 'text': 'A frozen backbone with a train-data classifier head is permitted transfer learning.', 'guide_corroborated': True}, {'id': 3, 'text': 'Official training samples may be omitted and train preprocessing/augmentation is permitted; no external data may be added.', 'guide_corroborated': True}, {'id': 4, 'date': '2026-07-27', 'text': 'Incorrectly labeled training samples may be deleted; preprocessing may do anything except add external data.', 'guide_corroborated': False}, {'id': 5, 'text': 'Test pseudo-labeling or any test-driven parameter update is prohibited.', 'guide_corroborated': True}, {'id': 6, 'text': 'All 1,458 current test images are scored, without public/private split.', 'guide_corroborated': False}, {'id': 7, 'text': 'Finalists will be verified on a new unseen test set.', 'guide_corroborated': False}, {'id': 8, 'text': 'Only three preliminary submissions are allowed.', 'guide_corroborated': True}, {'id': 9, 'date': '2026-07-22', 'text': 'Use 20 selected teams, superseding an earlier mention of 22.', 'guide_corroborated': True}, {'id': 10, 'text': 'Cleaning must be reproducible and explainable for verification.', 'guide_corroborated': True}], 'interpretation': 'Permission is not evidence: model disagreement only generates candidates; readable semantic rows require human confirmation.'}
LEADERBOARD_CONTEXT = {**provenance(), 'snapshot_date': '2026-07-27', 'source': 'user-supplied snapshot', 'scores': {'ranks_1_3': 100.0000, 'ranks_4_5': 99.9462, 'rank_6': 99.8925, 'ranks_7_8': 99.8387, 'ranks_9_13': 99.7850, 'rank_15': 99.7311, 'rank_20': 99.6179, 'rank_22': 99.5697, 'rank_50': 99.2454}, 'use': 'urgency and submission-budget context only', 'not_a_training_target': True, 'no_oof_to_leaderboard_mapping_claimed': True}
write_json(COMPETITION_CLARIFICATIONS, ARTIFACT_DIR / 'competition_clarification_record.json')
write_json(LEADERBOARD_CONTEXT, ARTIFACT_DIR / 'leaderboard_context.json')
print('RUN_MODE:', RUN_MODE, '| artifacts:', ARTIFACT_DIR)


RUN_MODE: SCREEN | artifacts: /kaggle/working/v9_dinov3_vit7b_frozen_drop_vs_relabel


## 6. Bounded artifact/cache/raw-train discovery

V7 and V8 discovery checks explicit known locations plus immediate Kaggle dataset roots and one named child. Raw train is resolved only in AUDIT mode. No broad storage scan and no test path are permitted.

In [2]:
V7_REQUIRED = ('run_manifest.json', 'experiment_decision.json', 'head_selection.json', 'feature_cache_manifest.json', 'data_and_fold_signature.json', 'model_and_preprocess_audit.json', 'preflight_report.json', 'fold_assignments.csv', 'fold0_head_metrics.json', 'heads/development_regularized_mlp_final_fold0_seed42_predictions.csv', 'heads/development_baseline_mlp_final_fold0_seed42_predictions.csv', 'assessment_selected_predictions_folds1_4.csv', 'assessment_baseline_predictions_folds1_4.csv', 'heads/development_regularized_mlp_final_fold0_seed42.pt')
V8_REQUIRED = ('run_manifest.json', 'control_reproduction.json', 'fold0_candidate_metrics.json', 'fold0_decision.json', 'final_adoption_decision.json')

def bounded_named_candidates(explicit, directory_name):
    candidates = [Path(path) for path in explicit]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for root in sorted(path for path in kaggle_input.iterdir() if path.is_dir()):
            candidates.extend((root, root / directory_name))
    candidates.append(KAGGLE_WORKING / directory_name)
    unique = []
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            unique.append(candidate)
            seen.add(key)
    return unique

def resolve_unique(name, candidates, required, directory_required=None):
    matches = [path for path in candidates if all((path / item).is_file() for item in required) and (directory_required is None or (path / directory_required).is_dir())]
    if len(matches) != 1:
        fail_closed(name + '_discovery', 'Expected one complete {} root; found {}.'.format(name, len(matches)), {'matches': [str(path) for path in matches]})
    if contains_forbidden_test_reference(str(matches[0])):
        fail_closed('test_access_guard', '{} root contains forbidden path component.'.format(name))
    return matches[0]

V7_ROOT = resolve_unique('v7', bounded_named_candidates(('/kaggle/input/datasets/riantamaputra/v7-results/v7_dinov3_vit7b_frozen_raw', '/kaggle/input/v7-results/v7_dinov3_vit7b_frozen_raw'), V7_DIRECTORY), V7_REQUIRED, 'feature_cache/shards')
V8_ROOT = resolve_unique('v8', bounded_named_candidates(('/kaggle/input/datasets/riantamaputra/v8-results/v8_dinov3_vit7b_frozen_cache_upgrade', '/kaggle/input/v8-results/v8_dinov3_vit7b_frozen_cache_upgrade'), V8_DIRECTORY), V8_REQUIRED)

V7_RUN = read_json(V7_ROOT / 'run_manifest.json')
V7_DECISION = read_json(V7_ROOT / 'experiment_decision.json')
V7_SELECTION = read_json(V7_ROOT / 'head_selection.json')
V7_CACHE_MANIFEST = read_json(V7_ROOT / 'feature_cache_manifest.json')
V7_DATA_SIGNATURE = read_json(V7_ROOT / 'data_and_fold_signature.json')
V7_MODEL_AUDIT = read_json(V7_ROOT / 'model_and_preprocess_audit.json')
V7_PREFLIGHT = read_json(V7_ROOT / 'preflight_report.json')
V7_FOLD0_METRICS = read_json(V7_ROOT / 'fold0_head_metrics.json')
V8_RUN = read_json(V8_ROOT / 'run_manifest.json')
V8_CONTROL_REPRODUCTION = read_json(V8_ROOT / 'control_reproduction.json')
V8_CANDIDATE_METRICS = read_json(V8_ROOT / 'fold0_candidate_metrics.json')
V8_FOLD0_DECISION = read_json(V8_ROOT / 'fold0_decision.json')
V8_FINAL_DECISION = read_json(V8_ROOT / 'final_adoption_decision.json')
for name, value in {'v7_run': V7_RUN, 'v7_decision': V7_DECISION, 'v7_selection': V7_SELECTION, 'v7_cache': V7_CACHE_MANIFEST, 'v7_data': V7_DATA_SIGNATURE, 'v7_model': V7_MODEL_AUDIT, 'v7_preflight': V7_PREFLIGHT, 'v7_fold0': V7_FOLD0_METRICS, 'v8_run': V8_RUN, 'v8_control': V8_CONTROL_REPRODUCTION, 'v8_metrics': V8_CANDIDATE_METRICS, 'v8_decision': V8_FOLD0_DECISION, 'v8_final': V8_FINAL_DECISION}.items():
    if contains_forbidden_test_reference(value):
        fail_closed('test_access_guard', 'Forbidden path in loaded {} artifact.'.format(name))
print('V7 root:', V7_ROOT)
print('V8 root:', V8_ROOT)


V7 root: /kaggle/input/datasets/riantamaputra/v7-results/v7_dinov3_vit7b_frozen_raw
V8 root: /kaggle/input/datasets/riantamaputra/v8-results/v8_dinov3_vit7b_frozen_cache_upgrade


## 7. Cache, folds, and sample-map validation

Every shard is checked for schema, range, metadata, dtype, finiteness, labels, folds, and ordered identities. AUDIT validates without retaining features; training modes retain only the final cached block. Complete selected/baseline OOF tables are aligned to the canonical mapping and re-scored.

In [3]:
try:
    selected_matches = [row for row in V7_FOLD0_METRICS if row.get('candidate') == CONTROL_NAME]
    last4_matches = [row for row in V7_FOLD0_METRICS if row.get('candidate') == 'last4_layer_weighted_gated']
    require(len(selected_matches) == 1 and V7_SELECTION.get('selected_head') == CONTROL_NAME, 'V7 selected control is absent or ambiguous.')
    require(len(last4_matches) == 1 and math.isclose(last4_matches[0]['macro_f1'], EXPECTED_V7_LAST4_MACRO_F1, abs_tol=1e-12, rel_tol=0.0) and last4_matches[0]['macro_f1'] < selected_matches[0]['macro_f1'], 'V7 last-four comparator evidence mismatch.')
    require(V7_RUN.get('experiment_name') == V7_DIRECTORY and V7_RUN.get('revision_marker') == V7_CACHE_REVISION and V7_RUN.get('feature_cache_complete') is True and V7_RUN.get('selected_head') == CONTROL_NAME and V7_RUN.get('preflight_contract_signature') == EXPECTED_PREFLIGHT_SIGNATURE, 'V7 run-manifest decision contract mismatch.')
    require(V7_SELECTION.get('schema_version') == 1 and V7_SELECTION.get('revision_marker') == V7_CACHE_REVISION and V7_SELECTION.get('development_fold') == DEVELOPMENT_FOLD and V7_SELECTION.get('assessment_folds_accessed_during_selection') == [] and math.isclose(V7_SELECTION.get('selected_fold0_macro_f1'), EXPECTED_V7_FOLD0_MACRO_F1, abs_tol=1e-12, rel_tol=0.0), 'V7 head-selection contract mismatch.')
    require(V7_DECISION.get('experiment') == V7_DIRECTORY and V7_DECISION.get('revision_marker') == V7_CACHE_REVISION and V7_DECISION.get('head_readout_result', {}).get('selected_head') == CONTROL_NAME and V7_DECISION.get('follow_up_launched') is False, 'V7 experiment decision mismatch.')
    V7_FOLD0_CONTROL = selected_matches[0]
    require(V7_RUN.get('feature_cache_signature') == CACHE_SIGNATURE_EXPECTED, 'V7 cache signature mismatch.')
    require(V7_MODEL_AUDIT.get('model_id') == MODEL_ID and V7_MODEL_AUDIT.get('resolved_revision') == MODEL_REVISION, 'V7 model identity mismatch.')
    require(V7_MODEL_AUDIT.get('trainable_parameter_count') == 0 and V7_MODEL_AUDIT.get('final_norm_is_frozen') is True, 'V7 backbone/final norm was not frozen.')
    require(math.isclose(V7_FOLD0_CONTROL['macro_f1'], EXPECTED_V7_FOLD0_MACRO_F1, abs_tol=1e-12, rel_tol=0.0), 'V7 fold-0 Macro F1 mismatch.')
    require(V7_FOLD0_CONTROL['total_errors'] == EXPECTED_V7_FOLD0_ERRORS and V7_FOLD0_CONTROL['recyclable_to_organic'] == EXPECTED_V7_FOLD0_R2O and V7_FOLD0_CONTROL['organic_to_recyclable'] == EXPECTED_V7_FOLD0_O2R, 'V7 fold-0 errors mismatch.')
    cache_identity = V7_CACHE_MANIFEST.get('cache_identity', {})
    require(V7_CACHE_MANIFEST.get('schema_version') == 2 and V7_CACHE_MANIFEST.get('revision_marker') == V7_CACHE_REVISION and V7_CACHE_MANIFEST.get('complete') is True, 'V7 cache schema/revision is incompatible.')
    require(V7_CACHE_MANIFEST.get('cache_signature') == CACHE_SIGNATURE_EXPECTED and V7_CACHE_MANIFEST.get('rows') == EXPECTED_ROWS, 'V7 cache identity/count mismatch.')
    require(tuple(V7_CACHE_MANIFEST.get('feature_tensor_shape', ())) == FEATURE_SHAPE, 'V7 cache shape mismatch.')
    require(cache_identity.get('schema_version') == 2 and cache_identity.get('revision_marker') == V7_CACHE_REVISION, 'V7 cache identity schema/revision mismatch.')
    require(cache_identity.get('model_id') == MODEL_ID and cache_identity.get('model_revision') == MODEL_REVISION and 'lvd1689m' in MODEL_ID and 'sat493m' not in MODEL_ID, 'Wrong DINOv3 source.')
    require(cache_identity.get('split') == 'official_raw_train' and cache_identity.get('image_size') == 256 and cache_identity.get('feature_view') == 'orig' and cache_identity.get('hidden_size') == HIDDEN_SIZE, 'V7 train/image/readout contract mismatch.')
    require(cache_identity.get('backbone_dtype') == 'float16' and cache_identity.get('feature_storage_dtype') == 'float16', 'V7 precision contract mismatch.')
    require(tuple(cache_identity.get('last_four_block_indices', ())) == LAST_BLOCKS and tuple(cache_identity.get('last_four_block_names', ())) == LAST_BLOCK_NAMES and cache_identity.get('pooled_layout') == ['cls', 'mean_true_patch_tokens'], 'V7 pooled representation mismatch.')
    require(cache_identity.get('per_token_normalization') == 'official pretrained DINOv3 final LayerNorm' and cache_identity.get('normalization_timing') == 'before CLS extraction and true-patch pooling', 'V7 final LayerNorm contract mismatch.')
    require(cache_identity.get('final_block_source') == 'model_output.last_hidden_state' and cache_identity.get('preflight_contract_signature') == EXPECTED_PREFLIGHT_SIGNATURE, 'V7 final-block/preflight contract mismatch.')
    require(cache_identity.get('fold_assignment_sha256') == FOLD_SIGNATURE_EXPECTED and cache_identity.get('ordered_image_ids_sha256') == IMAGE_IDS_SHA256_EXPECTED and cache_identity.get('labels_sha256') == LABELS_SHA256_EXPECTED and cache_identity.get('raw_data_sha256') == RAW_DATA_SHA256_EXPECTED, 'V7 data signatures mismatch.')
    require(V7_MODEL_AUDIT.get('hidden_size') == HIDDEN_SIZE and V7_MODEL_AUDIT.get('cached_representation') == 'official pretrained final LayerNorm per token before CLS/true-patch pooling', 'V7 model representation audit mismatch.')
    require(V7_PREFLIGHT.get('pass') is True and V7_PREFLIGHT.get('schema_version') == 2 and V7_PREFLIGHT.get('preflight_contract_signature') == EXPECTED_PREFLIGHT_SIGNATURE, 'V7 strict preflight mismatch.')
    final_norm_equivalence = V7_PREFLIGHT.get('stages', {}).get('B_sharded_token_memory_audit', {}).get('official_final_norm_equivalence')
    require(isinstance(final_norm_equivalence, list) and len(final_norm_equivalence) == 2 and all(item.get('pass') is True for item in final_norm_equivalence), 'V7 final-LayerNorm equivalence audit failed.')

    fold_assignments = pd.read_csv(V7_ROOT / 'fold_assignments.csv')
    require(list(fold_assignments.columns) == ['image_id', 'label', 'class_name', 'fold'] and len(fold_assignments) == EXPECTED_ROWS, 'Fold mapping schema/count mismatch.')
    require(fold_assignments['image_id'].is_unique and fold_assignments['image_id'].astype(str).tolist() == sorted(fold_assignments['image_id'].astype(str).tolist()), 'Sample mapping order/uniqueness failed.')
    require(not fold_assignments['image_id'].astype(str).map(contains_forbidden_test_reference).any(), 'Forbidden path in train mapping.')
    observed_counts = {int(key): int(value) for key, value in fold_assignments['label'].value_counts().sort_index().items()}
    require(observed_counts == EXPECTED_CLASS_COUNTS and set(fold_assignments['fold'].astype(int)) == set(range(N_FOLDS)), 'Class/fold counts mismatch.')
    require(fold_assignments['label'].astype(int).map(CLASS_MAPPING).tolist() == fold_assignments['class_name'].astype(str).tolist(), 'Class mapping mismatch.')
    require(sha256_json(fold_assignments['image_id'].astype(str).tolist()) == IMAGE_IDS_SHA256_EXPECTED, 'Ordered image hash mismatch.')
    require(sha256_json(fold_assignments['label'].astype(int).tolist()) == LABELS_SHA256_EXPECTED, 'Label hash mismatch.')
    fold_records = [{'image_id': str(row.image_id), 'label': int(row.label), 'fold': int(row.fold)} for row in fold_assignments.itertuples(index=False)]
    require(sha256_json(fold_records) == FOLD_SIGNATURE_EXPECTED, 'Fold signature recomputation failed.')
    require(V7_DATA_SIGNATURE.get('fold_assignment_sha256') == FOLD_SIGNATURE_EXPECTED and V7_DATA_SIGNATURE.get('raw_data_sha256') == RAW_DATA_SHA256_EXPECTED, 'V7 data-signature artifact mismatch.')
    require(V7_DATA_SIGNATURE.get('raw_class_counts') == {str(key): value for key, value in EXPECTED_CLASS_COUNTS.items()} and V7_DATA_SIGNATURE.get('official_class_codes') == {str(key): value for key, value in CLASS_MAPPING.items()}, 'V7 data counts/mapping audit mismatch.')

    def load_complete_oof(fold0_name, folds1_4_name, expected_candidate):
        frame = pd.concat((pd.read_csv(V7_ROOT / fold0_name), pd.read_csv(V7_ROOT / folds1_4_name)), ignore_index=True).sort_values('row_index').reset_index(drop=True)
        require(len(frame) == EXPECTED_ROWS and frame['row_index'].astype(int).tolist() == list(range(EXPECTED_ROWS)), '{} OOF coverage failed.'.format(expected_candidate))
        require(frame['image_id'].astype(str).tolist() == fold_assignments['image_id'].astype(str).tolist(), '{} OOF identities failed.'.format(expected_candidate))
        require(frame['label'].astype(int).tolist() == fold_assignments['label'].astype(int).tolist() and frame['fold'].astype(int).tolist() == fold_assignments['fold'].astype(int).tolist(), '{} OOF labels/folds failed.'.format(expected_candidate))
        require(set(frame['candidate'].astype(str)) == {expected_candidate}, '{} candidate identity failed.'.format(expected_candidate))
        probabilities = normalize_probabilities(frame[['prob_0', 'prob_1', 'prob_2']].to_numpy())
        require(np.array_equal(probabilities.argmax(axis=1), frame['prediction'].astype(int).to_numpy()), '{} stored predictions mismatch probabilities.'.format(expected_candidate))
        return frame, probabilities

    selected_oof, selected_oof_probabilities = load_complete_oof('heads/development_regularized_mlp_final_fold0_seed42_predictions.csv', 'assessment_selected_predictions_folds1_4.csv', CONTROL_NAME)
    baseline_oof, baseline_oof_probabilities = load_complete_oof('heads/development_baseline_mlp_final_fold0_seed42_predictions.csv', 'assessment_baseline_predictions_folds1_4.csv', 'baseline_mlp_final')
    oof_labels = fold_assignments['label'].astype(int).to_numpy()
    selected_oof_metrics = metric_bundle(oof_labels, selected_oof_probabilities)
    baseline_oof_metrics = metric_bundle(oof_labels, baseline_oof_probabilities)
    require(math.isclose(selected_oof_metrics['macro_f1'], EXPECTED_V7_FIVEFOLD_MACRO_F1, abs_tol=1e-12, rel_tol=0.0), 'V7 five-fold Macro F1 mismatch.')
    require(selected_oof_metrics['total_errors'] == EXPECTED_V7_FIVEFOLD_ERRORS and selected_oof_metrics['recyclable_to_organic'] + selected_oof_metrics['organic_to_recyclable'] == EXPECTED_V7_FIVEFOLD_RO_ERRORS, 'V7 five-fold errors mismatch.')
    require(math.isclose(selected_oof_metrics['classification_report']['Electronic']['f1-score'], EXPECTED_V7_FIVEFOLD_E_F1, abs_tol=1e-12, rel_tol=0.0), 'V7 five-fold Electronic F1 mismatch.')
    confirmation_mask = fold_assignments['fold'].astype(int).to_numpy() != DEVELOPMENT_FOLD
    confirmation_metrics = metric_bundle(oof_labels[confirmation_mask], selected_oof_probabilities[confirmation_mask])
    require(math.isclose(confirmation_metrics['macro_f1'], EXPECTED_V7_CONFIRMATION_MACRO_F1, abs_tol=1e-12, rel_tol=0.0), 'V7 folds 1-4 Macro F1 mismatch.')
    require(confirmation_metrics['total_errors'] == EXPECTED_V7_CONFIRMATION_TOTAL_ERRORS and confirmation_metrics['recyclable_to_organic'] + confirmation_metrics['organic_to_recyclable'] == EXPECTED_V7_CONFIRMATION_RO_ERRORS and math.isclose(confirmation_metrics['classification_report']['Electronic']['f1-score'], EXPECTED_V7_CONFIRMATION_E_F1, abs_tol=1e-12, rel_tol=0.0), 'V7 folds 1-4 error/Electronic evidence mismatch.')
    selected_predictions = selected_oof_probabilities.argmax(axis=1)
    baseline_predictions = baseline_oof_probabilities.argmax(axis=1)
    consensus_wrong = (selected_predictions == baseline_predictions) & (selected_predictions != oof_labels)
    require(int(consensus_wrong.sum()) == EXPECTED_V7_CONSENSUS_WRONG, 'V7 selected/baseline consensus count mismatch.')
    OOF_EVIDENCE = pd.DataFrame({'sample_index': np.arange(EXPECTED_ROWS), 'relative_path': fold_assignments['image_id'].astype(str), 'fold': fold_assignments['fold'].astype(int), 'original_label': oof_labels, 'oof_predicted_label': selected_predictions, 'baseline_predicted_label': baseline_predictions, 'oof_original_probability': selected_oof_probabilities[np.arange(EXPECTED_ROWS), oof_labels], 'oof_top_probability': selected_oof_probabilities.max(axis=1), 'oof_margin': np.sort(selected_oof_probabilities, axis=1)[:, -1] - np.sort(selected_oof_probabilities, axis=1)[:, -2], 'baseline_original_probability': baseline_oof_probabilities[np.arange(EXPECTED_ROWS), oof_labels], 'baseline_top_probability': baseline_oof_probabilities.max(axis=1), 'baseline_margin': np.sort(baseline_oof_probabilities, axis=1)[:, -1] - np.sort(baseline_oof_probabilities, axis=1)[:, -2], 'selected_baseline_consensus': consensus_wrong, 'selected_wrong': selected_predictions != oof_labels, 'baseline_wrong': baseline_predictions != oof_labels})

    blend_metrics = V8_CANDIDATE_METRICS['candidates']['ro_specialist_blend75']
    full_metrics = V8_CANDIDATE_METRICS['candidates']['ro_specialist_full']
    require(V8_RUN.get('experiment_name') == V8_DIRECTORY and V8_RUN.get('revision_marker') == V8_REVISION and V8_RUN.get('v7_cache_signature') == CACHE_SIGNATURE_EXPECTED and V8_RUN.get('fold_assignment_sha256') == FOLD_SIGNATURE_EXPECTED and V8_RUN.get('model_id_inherited_from_v7') == MODEL_ID and V8_RUN.get('model_revision_inherited_from_v7') == MODEL_REVISION, 'V8 run-manifest provenance mismatch.')
    require(V8_CONTROL_REPRODUCTION.get('pass') is True and V8_CONTROL_REPRODUCTION.get('absolute_macro_f1_difference') == 0.0, 'V8 control did not reproduce V7 exactly.')
    require(math.isclose(blend_metrics['macro_f1'], EXPECTED_V8_BLEND_MACRO_F1, abs_tol=1e-12, rel_tol=0.0) and math.isclose(full_metrics['macro_f1'], EXPECTED_V8_FULL_MACRO_F1, abs_tol=1e-12, rel_tol=0.0), 'V8 candidate metrics mismatch.')
    require(blend_metrics['total_errors'] == 65 and full_metrics['total_errors'] == 64 and V8_FOLD0_DECISION.get('pass') is False and V8_FINAL_DECISION.get('adopt_v8') is False, 'V8 rejection evidence mismatch.')

    checkpoint = torch.load(V7_ROOT / 'heads/development_regularized_mlp_final_fold0_seed42.pt', map_location='cpu', weights_only=False)
    V7_CONTROL_METADATA = checkpoint.get('metadata')
    state = checkpoint.get('state_dict')
    require(isinstance(V7_CONTROL_METADATA, dict) and isinstance(state, dict), 'V7 control checkpoint contract missing.')
    require(not contains_forbidden_test_reference(V7_CONTROL_METADATA), 'Forbidden path in V7 control checkpoint metadata.')
    expected_control_metadata = {'candidate': CONTROL_NAME, 'seed': SEED, 'cache_signature': CACHE_SIGNATURE_EXPECTED, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'gradient_clip_max_norm': GRAD_CLIP}
    require(all(V7_CONTROL_METADATA.get(key) == value for key, value in expected_control_metadata.items()), 'V7 control checkpoint recipe/provenance mismatch.')
    require(V7_CONTROL_METADATA.get('candidate_spec', {}).get('dropout') == DROPOUT and V7_CONTROL_METADATA.get('candidate_spec', {}).get('label_smoothing') == LABEL_SMOOTHING, 'V7 control recipe mismatch.')
    expected_state_shapes = {'network.0.weight': (2 * HIDDEN_SIZE,), 'network.0.bias': (2 * HIDDEN_SIZE,), 'network.1.weight': (HEAD_HIDDEN, 2 * HIDDEN_SIZE), 'network.1.bias': (HEAD_HIDDEN,), 'network.4.weight': (N_CLASSES, HEAD_HIDDEN), 'network.4.bias': (N_CLASSES,)}
    require(set(state) == set(expected_state_shapes) and all(tuple(state[key].shape) == shape for key, shape in expected_state_shapes.items()), 'V7 control checkpoint architecture mismatch.')
    del checkpoint, state

    manifest_shards = sorted(V7_CACHE_MANIFEST.get('shards', []), key=lambda row: int(row['start']))
    require(len(manifest_shards) == V7_CACHE_MANIFEST.get('shard_count') == 104, 'V7 shard count mismatch.')
    expected_start = 0
    expected_names = []
    for shard_index, record in enumerate(manifest_shards):
        start, end = int(record['start']), int(record['end'])
        expected_name = 'train_{:05d}_{:05d}.pt'.format(start, end)
        require(record.get('shard_index') == shard_index and record.get('shard_count') == len(manifest_shards) and start == expected_start and end > start and int(record['rows']) == end - start and Path(record.get('path', '')).name == expected_name, 'Shard metadata/coverage gap or overlap.')
        expected_names.append(expected_name)
        expected_start = end
    require(expected_start == EXPECTED_ROWS, 'Shard coverage incomplete.')
    shards_dir = V7_ROOT / 'feature_cache/shards'
    require(sorted(path.name for path in shards_dir.glob('*.pt')) == sorted(expected_names), 'Shard files mismatch manifest.')
    cached_final_features = torch.empty((EXPECTED_ROWS, 2, HIDDEN_SIZE), dtype=torch.float16) if RUN_MODE != 'AUDIT' else None
    cached_labels = torch.tensor(oof_labels, dtype=torch.long)
    cached_folds = torch.tensor(fold_assignments['fold'].astype(int).to_numpy(), dtype=torch.long)
    for record, name in zip(manifest_shards, expected_names):
        start, end = int(record['start']), int(record['end'])
        expected = fold_assignments.iloc[start:end]
        expected_metadata = {'schema_version': 2, 'revision_marker': V7_CACHE_REVISION, 'cache_signature': CACHE_SIGNATURE_EXPECTED, 'start': start, 'end': end, 'rows': end - start, 'image_ids_sha256': sha256_json(expected['image_id'].astype(str).tolist()), 'labels_sha256': sha256_json(expected['label'].astype(int).tolist()), 'folds_sha256': sha256_json(expected['fold'].astype(int).tolist()), 'feature_shape': [end - start, 4, 2, HIDDEN_SIZE], 'feature_dtype': 'float16'}
        payload = torch.load(shards_dir / name, map_location='cpu', weights_only=False)
        require(isinstance(payload, dict) and payload.get('metadata') == expected_metadata, 'Shard metadata mismatch in {}.'.format(name))
        features, labels, folds, image_ids = payload.get('features'), payload.get('labels'), payload.get('folds'), payload.get('image_ids')
        require(torch.is_tensor(features) and tuple(features.shape) == (end - start, 4, 2, HIDDEN_SIZE) and features.dtype == torch.float16 and torch.isfinite(features).all().item(), 'Invalid features in {}.'.format(name))
        require(torch.is_tensor(labels) and labels.dtype == torch.long and torch.is_tensor(folds) and folds.dtype == torch.long, 'Invalid label/fold tensors in {}.'.format(name))
        require(torch.equal(labels, torch.tensor(expected['label'].astype(int).to_numpy(), dtype=torch.long)) and torch.equal(folds, torch.tensor(expected['fold'].astype(int).to_numpy(), dtype=torch.long)), 'Shard label/fold mismatch in {}.'.format(name))
        require(image_ids == expected['image_id'].astype(str).tolist(), 'Shard identity mismatch in {}.'.format(name))
        if cached_final_features is not None:
            cached_final_features[start:end].copy_(features[:, -1])
        del payload, features, labels, folds, image_ids
    if cached_final_features is not None:
        require(torch.isfinite(cached_final_features).all().item(), 'Retained final cache is non-finite.')

    V7_ARTIFACT_AUDIT = {**provenance(), 'pass': True, 'v7_root': str(V7_ROOT), 'fold0_control': V7_FOLD0_CONTROL, 'fold0_last4_comparator': last4_matches[0], 'folds1_4_selected': confirmation_metrics, 'fivefold_selected': selected_oof_metrics, 'fivefold_baseline': baseline_oof_metrics, 'selected_baseline_consensus_wrong': int(consensus_wrong.sum()), 'complete_oof_rows': EXPECTED_ROWS, 'control_checkpoint_metadata': V7_CONTROL_METADATA}
    V8_ARTIFACT_AUDIT = {**provenance(), 'pass': True, 'v8_root': str(V8_ROOT), 'control_reproduction': V8_CONTROL_REPRODUCTION, 'blend': {'macro_f1': blend_metrics['macro_f1'], 'total_errors': blend_metrics['total_errors'], 'ro_errors': blend_metrics['recyclable_to_organic'] + blend_metrics['organic_to_recyclable']}, 'full': {'macro_f1': full_metrics['macro_f1'], 'total_errors': full_metrics['total_errors'], 'ro_errors': full_metrics['recyclable_to_organic'] + full_metrics['organic_to_recyclable']}, 'confirmation_ran': False, 'retained_v7': True}
    CACHE_AUDIT = {**provenance(), 'pass': True, 'rows': EXPECTED_ROWS, 'feature_shape': list(FEATURE_SHAPE), 'retained_shape': list(cached_final_features.shape) if cached_final_features is not None else None, 'dtype': 'float16', 'shards': len(manifest_shards), 'complete_nonoverlapping_coverage': True, 'official_final_layernorm_before_pooling': True, 'backbone_frozen': True}
    FAMILY_EVALUATION = {'audited_exclusion': {'expected_mechanism': 'remove a tiny amount of contradictory supervision', 'project_evidence': '230 same-alternative OOF errors plus suspicious gallery rows', 'leakage_risk': 'low with OOF-only evidence and unchanged validation', 'manual_review_burden': 'bounded but material', 'reproducibility': 'exact reviewed manifest and hashes', 'compute_cost': 'one cache-only head per enabled candidate', 'fivefold_survival': 'plausible but unproven', 'second_submission_diversity': 'high because the training subset changes', 'deadline_risk': 'manual review quality'}, 'targeted_ro_mixup': {'expected_mechanism': 'smooth the R/O feature manifold', 'project_evidence': 'weak after V8 specialist regression', 'leakage_risk': 'low', 'manual_review_burden': 'none', 'reproducibility': 'high', 'compute_cost': 'cache-only', 'fivefold_survival': 'low-to-uncertain', 'second_submission_diversity': 'moderate', 'deadline_risk': 'synthetic ambiguity'}, 'v7_multiseed_average': {'expected_mechanism': 'reduce head variance', 'project_evidence': 'no evidence that variance dominates label noise', 'leakage_risk': 'low', 'manual_review_burden': 'none', 'reproducibility': 'high', 'compute_cost': 'several extra heads', 'fivefold_survival': 'moderate', 'second_submission_diversity': 'low', 'deadline_risk': 'near-duplicate upload'}, 'existing_v7_head_ensemble': {'expected_mechanism': 'exploit fixed error complementarity', 'project_evidence': 'baseline is lower-scoring and overconfident', 'leakage_risk': 'low', 'manual_review_burden': 'none', 'reproducibility': 'high', 'compute_cost': 'minimal', 'fivefold_survival': 'low without robust complementarity', 'second_submission_diversity': 'low', 'deadline_risk': 'weak value'}, 'no_v9_experiment': {'expected_mechanism': 'remove research risk and prepare inference', 'project_evidence': 'appropriate if review yields no defensible drops', 'leakage_risk': 'none', 'manual_review_burden': 'none', 'reproducibility': 'highest', 'compute_cost': 'none', 'fivefold_survival': 'not applicable', 'second_submission_diversity': 'none', 'deadline_risk': 'no new candidate'}}
    APPROACH_DECISION = {**provenance(), 'decision_timing': 'before V9 training or validation scores', 'selected_family': 'cross-fitted audited drop-versus-relabel treatment with unchanged V7 joint head', 'audited_drop_and_relabel_accepted_conditionally': True, 'condition': 'Only two objective duplicate conflicts may be dropped automatically; exactly ten locked human-confirmed Tier-B rows may be compared as drop-only versus training-only relabel.', 'expected_mechanism': 'compare removing contradictory supervision against correcting it while preserving complete official validation', 'evidence': {'v7_fivefold_macro_f1': selected_oof_metrics['macro_f1'], 'v7_errors': selected_oof_metrics['total_errors'], 'v7_ro_errors': selected_oof_metrics['recyclable_to_organic'] + selected_oof_metrics['organic_to_recyclable'], 'v7_consensus_wrong': int(consensus_wrong.sum()), 'v8_candidates_regressed': True}, 'family_evaluation': FAMILY_EVALUATION, 'alternatives': {'targeted_ro_mixup': {'decision': 'reject', 'reason': 'synthetic ambiguity and no project evidence after V8 specialist regression'}, 'v7_multiseed_average': {'decision': 'reject', 'reason': 'near-duplicate model that does not address contradictory supervision'}, 'existing_v7_head_ensemble': {'decision': 'reject', 'reason': 'baseline is lower-scoring and overconfident; robust complementary gain is not established'}, 'no_v9_experiment': {'decision': 'fallback', 'reason': 'use if AUDIT yields no defensible reviewed drops'}}, 'risks': ['human semantic review may be irreproducible', 'R/O semantics remain ambiguous', 'small gains may be fold noise', 'deletion may reduce useful hard-example coverage'], 'official_clarification_used': 'train preprocessing may include deletion or relabeling; model disagreement alone is not treated as truth'}
    write_json(V7_ARTIFACT_AUDIT, ARTIFACT_DIR / 'v7_artifact_audit.json')
    write_json(V8_ARTIFACT_AUDIT, ARTIFACT_DIR / 'v8_artifact_audit.json')
    write_json(CACHE_AUDIT, ARTIFACT_DIR / 'cache_compatibility_audit.json')
    write_json(APPROACH_DECISION, ARTIFACT_DIR / 'approach_decision.json')
except Exception as exc:
    fail_closed('evidence_cache_audit', str(exc))
print('V7 five-fold:', selected_oof_metrics['macro_f1'], '| consensus wrong:', int(consensus_wrong.sum()))


V7 five-fold: 0.9901360067264733 | consensus wrong: 230


## 8. AUDIT candidate generation and contact sheets

AUDIT uses only cross-fitted V7 predictions and official raw-train images. Exact byte-duplicate conflicts and objectively unreadable/blank images form Tier A. Strict selected/baseline consensus forms a bounded Tier-B review queue, but every semantic row is prefilled KEEP. All other model-only or ambiguous errors are Tier C and remain KEEP. No V9 head is trained.

In [4]:
AUDIT_MODE_DECISION = None
CONTACT_SHEET_FILES = []
if RUN_MODE == 'AUDIT':
    explicit_train = Path('/kaggle/input/datasets/riantamaputra/bdc-2026-waste-classification-dataset/train')
    train_candidates = [explicit_train, Path('/kaggle/input/bdc-2026-waste-classification-dataset/train')]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        train_candidates.extend(root / 'train' for root in sorted(path for path in kaggle_input.iterdir() if path.is_dir()))
    train_matches = []
    for root in train_candidates:
        if root.is_dir() and all((root / '{}_{}'.format(label, CLASS_MAPPING[label])).is_dir() for label in CLASS_MAPPING):
            train_matches.append(root)
    train_matches = list(dict.fromkeys(str(path) for path in train_matches))
    if len(train_matches) != 1:
        fail_closed('raw_train_discovery', 'Expected one official raw-train root; found {}.'.format(len(train_matches)), {'matches': train_matches})
    RAW_TRAIN_ROOT = Path(train_matches[0])
    require(RAW_TRAIN_ROOT.name.lower() == 'train' and not contains_forbidden_test_reference(str(RAW_TRAIN_ROOT)), 'Raw audit root is not train-only.')
    raw_paths = {}
    for label in CLASS_MAPPING:
        class_dir = RAW_TRAIN_ROOT / '{}_{}'.format(label, CLASS_MAPPING[label])
        for path in sorted(item for item in class_dir.rglob('*') if item.is_file() and item.suffix.lower() in TRAIN_EXTENSIONS):
            relative = path.relative_to(RAW_TRAIN_ROOT).as_posix()
            require(relative not in raw_paths and not contains_forbidden_test_reference(relative), 'Duplicate/forbidden raw path.')
            raw_paths[relative] = path
    require(set(raw_paths) == set(fold_assignments['image_id'].astype(str)), 'Raw-train mapping differs from validated V7 mapping.')

    Image.MAX_IMAGE_PIXELS = 250_000_000
    quality_rows = []
    for sample_index, row in enumerate(fold_assignments.itertuples(index=False)):
        relative = str(row.image_id)
        path = raw_paths[relative]
        file_hash = sha256_file(path)
        readable = True
        error = ''
        width = height = 0
        channel_std = dynamic_range = float('nan')
        try:
            with Image.open(path) as image:
                width, height = image.size
                image.draft('RGB', (AUDIT_THUMBNAIL_SIDE, AUDIT_THUMBNAIL_SIDE))
                image = ImageOps.exif_transpose(image).convert('RGB')
                image.thumbnail((AUDIT_THUMBNAIL_SIDE, AUDIT_THUMBNAIL_SIDE), Image.Resampling.LANCZOS)
                pixels = np.asarray(image, dtype=np.uint8)
            require(pixels.ndim == 3 and pixels.shape[2] == 3 and pixels.size > 0, 'decoded RGB shape invalid')
            channel_std = float(pixels.astype(np.float32).std())
            dynamic_range = int(pixels.max()) - int(pixels.min())
        except Exception as exc:
            readable = False
            error = '{}: {}'.format(type(exc).__name__, str(exc)[:200])
        quality_rows.append({'sample_index': sample_index, 'file_sha256': file_hash, 'file_size_bytes': path.stat().st_size, 'raw_readable': readable, 'width': width, 'height': height, 'thumbnail_std': channel_std, 'thumbnail_dynamic_range': dynamic_range, 'decode_error': error})
    quality = pd.DataFrame(quality_rows)
    raw_signature_records = [{'image_id': str(row.image_id), 'label': int(row.label), 'file_size_bytes': int(quality.iloc[index]['file_size_bytes'])} for index, row in enumerate(fold_assignments.itertuples(index=False))]
    require(sha256_json(raw_signature_records) == RAW_DATA_SHA256_EXPECTED, 'Raw-train file-size signature differs from V7 evidence.')
    duplicate_labels = pd.DataFrame({'file_sha256': quality['file_sha256'], 'label': fold_assignments['label'].astype(int)}).groupby('file_sha256')['label'].nunique()
    conflict_hashes = set(duplicate_labels[duplicate_labels > 1].index)
    quality['exact_duplicate_conflict'] = quality['file_sha256'].isin(conflict_hashes)
    quality['objective_unusable'] = (~quality['raw_readable']) | ((quality['thumbnail_std'] <= BLANK_STD_MAX) & (quality['thumbnail_dynamic_range'] <= BLANK_RANGE_MAX)) | quality['exact_duplicate_conflict']
    evidence = OOF_EVIDENCE.merge(quality, on='sample_index', validate='one_to_one')
    semantic_queue = evidence['selected_baseline_consensus'] & (evidence['oof_top_probability'] >= SEMANTIC_QUEUE['selected_top_min']) & (evidence['oof_original_probability'] <= SEMANTIC_QUEUE['selected_original_max']) & (evidence['oof_margin'] >= SEMANTIC_QUEUE['selected_margin_min']) & (evidence['baseline_top_probability'] >= SEMANTIC_QUEUE['baseline_top_min']) & (evidence['baseline_original_probability'] <= SEMANTIC_QUEUE['baseline_original_max'])
    error_union = evidence['selected_wrong'] | evidence['baseline_wrong']
    evidence['candidate_tier'] = np.select((evidence['objective_unusable'], semantic_queue, error_union), (TIER_A, TIER_B, TIER_C), default=TIER_KEEP)
    evidence['is_ro_mismatch'] = evidence.apply(lambda row: {int(row.original_label), int(row.oof_predicted_label)} == {0, 2}, axis=1)
    evidence['is_electronic_mismatch'] = evidence.apply(lambda row: int(row.original_label) == 1 or int(row.oof_predicted_label) == 1, axis=1)
    evidence['manual_review_priority'] = 4.0 * evidence['selected_baseline_consensus'].astype(float) + 2.0 * evidence['selected_wrong'].astype(float) + evidence['oof_top_probability'] + evidence['oof_margin'] - evidence['oof_original_probability'] + 0.5 * evidence['is_electronic_mismatch'].astype(float) - 0.5 * evidence['is_ro_mismatch'].astype(float)
    candidates = evidence[evidence['candidate_tier'] != TIER_KEEP].copy().sort_values(['candidate_tier', 'manual_review_priority', 'sample_index'], ascending=[True, False, True]).reset_index(drop=True)
    now = datetime.now(timezone.utc).isoformat()
    candidates['reviewed_label'] = candidates['original_label'].astype(int)
    candidates['review_action'] = np.where(candidates['candidate_tier'] == TIER_A, 'DROP_CORRUPT_OR_UNUSABLE', np.where(candidates['candidate_tier'] == TIER_C, 'AMBIGUOUS_KEEP', 'KEEP'))
    candidates['review_reason'] = np.where(candidates['candidate_tier'] == TIER_A, np.where(candidates['exact_duplicate_conflict'], 'EXACT_BYTE_DUPLICATE_CONFLICT_ALL_COPIES_EXCLUDED', np.where(~candidates['raw_readable'], 'RAW_IMAGE_UNREADABLE', 'OBJECTIVELY_BLANK_OR_UNUSABLE')), np.where(candidates['candidate_tier'] == TIER_C, 'AMBIGUOUS_OR_MODEL_ONLY_EVIDENCE_KEEP', 'UNREVIEWED_SEMANTIC_CANDIDATE_DEFAULT_KEEP'))
    candidates['reviewer'] = np.where(candidates['candidate_tier'] == TIER_A, 'V9_OBJECTIVE_AUDIT', '')
    candidates['review_timestamp'] = np.where(candidates['candidate_tier'] == TIER_A, now, '')
    candidates['evidence_source'] = np.where(candidates['candidate_tier'] == TIER_A, 'RAW_TRAIN_OBJECTIVE_AUDIT', np.where(candidates['candidate_tier'] == TIER_B, 'V7_SELECTED_BASELINE_OOF_CONSENSUS', 'V7_CROSSFITTED_OOF_DIAGNOSTIC'))
    ranking_columns = list(REVIEW_REQUIRED_COLUMNS) + ['baseline_predicted_label', 'baseline_original_probability', 'baseline_top_probability', 'baseline_margin', 'selected_wrong', 'baseline_wrong', 'is_ro_mismatch', 'is_electronic_mismatch', 'manual_review_priority', 'raw_readable', 'width', 'height', 'thumbnail_std', 'thumbnail_dynamic_range', 'file_sha256', 'exact_duplicate_conflict', 'objective_unusable']
    candidate_ranking = candidates[ranking_columns]
    review_manifest = candidates[list(REVIEW_REQUIRED_COLUMNS)]
    write_csv(candidate_ranking, ARTIFACT_DIR / 'cleaning_candidate_ranking.csv')
    write_csv(review_manifest, ARTIFACT_DIR / 'cleaning_review_manifest.csv')
    candidate_contract_hash = candidate_contract_sha256(pd.read_csv(ARTIFACT_DIR / 'cleaning_candidate_ranking.csv'))
    figure, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for axis, column, title in zip(axes, ('oof_original_probability', 'oof_top_probability', 'oof_margin'), ('Original-label probability', 'Top probability', 'Top1-top2 margin')):
        if len(candidate_ranking):
            axis.hist(candidate_ranking[column].astype(float), bins=30)
        else:
            axis.text(0.5, 0.5, 'No candidates', ha='center', va='center')
        axis.set_title(title)
    figure.tight_layout()
    figure.savefig(ARTIFACT_DIR / 'cleaning_candidate_confidence_distributions.png', dpi=160, bbox_inches='tight')
    plt.close(figure)

    grouped = list(candidate_ranking.sort_values('manual_review_priority', ascending=False).groupby(['candidate_tier', 'original_label', 'oof_predicted_label'], sort=True))[:MAX_CONTACT_SHEETS]
    for page, ((tier, original, predicted), group) in enumerate(grouped, start=1):
        page_rows = group.head(CONTACT_SHEET_IMAGES)
        columns = 5
        rows = math.ceil(len(page_rows) / columns)
        figure, axes = plt.subplots(rows, columns, figsize=(15, 3.2 * rows))
        axes = np.asarray(axes).reshape(-1)
        for axis in axes:
            axis.axis('off')
        for axis, item in zip(axes, page_rows.itertuples(index=False)):
            try:
                with Image.open(raw_paths[item.relative_path]) as image:
                    image.draft('RGB', (AUDIT_THUMBNAIL_SIDE, AUDIT_THUMBNAIL_SIDE))
                    image = ImageOps.exif_transpose(image).convert('RGB')
                    image.thumbnail((AUDIT_THUMBNAIL_SIDE, AUDIT_THUMBNAIL_SIDE), Image.Resampling.LANCZOS)
                    axis.imshow(image)
            except Exception:
                axis.text(0.5, 0.5, 'UNREADABLE', ha='center', va='center')
            axis.set_title('#{} {}→{}\np={:.3f} orig={:.3f}\n{}'.format(item.sample_index, item.original_label, item.oof_predicted_label, item.oof_top_probability, item.oof_original_probability, item.relative_path), fontsize=7)
            axis.axis('off')
        figure.suptitle('{} | {} → {}'.format(tier, CLASS_MAPPING[int(original)], CLASS_MAPPING[int(predicted)]))
        figure.tight_layout()
        filename = 'contact_{:02d}_{}_{}_to_{}.png'.format(page, re.sub('[^A-Za-z0-9]+', '_', tier).strip('_').lower(), original, predicted)
        figure.savefig(ARTIFACT_DIR / filename, dpi=140, bbox_inches='tight')
        plt.close(figure)
        CONTACT_SHEET_FILES.append(filename)
    if not CONTACT_SHEET_FILES:
        figure, axis = plt.subplots(figsize=(8, 2))
        axis.text(0.5, 0.5, 'No cleaning candidates generated', ha='center', va='center')
        axis.axis('off')
        figure.savefig(ARTIFACT_DIR / 'contact_00_no_candidates.png', dpi=140, bbox_inches='tight')
        plt.close(figure)
        CONTACT_SHEET_FILES.append('contact_00_no_candidates.png')

    confidence_summary = {'rows': len(candidate_ranking)}
    if len(candidate_ranking):
        confidence_summary.update({column: {'minimum': float(candidate_ranking[column].min()), 'median': float(candidate_ranking[column].median()), 'mean': float(candidate_ranking[column].mean()), 'maximum': float(candidate_ranking[column].max())} for column in ('oof_original_probability', 'oof_top_probability', 'oof_margin')})
    CLEANING_CANDIDATE_SUMMARY = {**provenance(), 'raw_train_root': str(RAW_TRAIN_ROOT), 'raw_data_sha256_recomputed': sha256_json(raw_signature_records), 'candidate_contract_sha256': candidate_contract_hash, 'ranking_file_sha256': sha256_file(ARTIFACT_DIR / 'cleaning_candidate_ranking.csv'), 'rows': len(candidate_ranking), 'tier_counts': candidate_ranking['candidate_tier'].value_counts().to_dict(), 'pair_counts': candidate_ranking.groupby(['original_label', 'oof_predicted_label']).size().rename('rows').reset_index().to_dict(orient='records'), 'objective_drop_count': int((candidate_ranking['candidate_tier'] == TIER_A).sum()), 'semantic_review_count': int((candidate_ranking['candidate_tier'] == TIER_B).sum()), 'semantic_thresholds': SEMANTIC_QUEUE, 'confidence_summary': confidence_summary, 'oof_sources': ['V7 regularized_mlp_final', 'V7 baseline_mlp_final'], 'independent_backbone_oof_available': False, 'contact_sheets': CONTACT_SHEET_FILES, 'semantic_default_action': 'KEEP', 'relabeling_supported': True}
    AUDIT_MODE_DECISION = {**provenance(), 'pass': True, 'training_ran': False, 'next_step': 'Run SCREEN. The notebook will apply the locked 10 human-confirmed relabel decisions and preserve all ambiguous rows.', 'candidate_contract_sha256': candidate_contract_hash, 'review_required': int((candidate_ranking['candidate_tier'] == TIER_B).sum()) > 0}
    write_json(CLEANING_CANDIDATE_SUMMARY, ARTIFACT_DIR / 'cleaning_candidate_summary.json')
    write_json(AUDIT_MODE_DECISION, ARTIFACT_DIR / 'audit_mode_decision.json')
    print('AUDIT complete; no heads trained. Review rows:', len(review_manifest))
else:
    print('AUDIT image decoding skipped in', RUN_MODE)


AUDIT image decoding skipped in SCREEN


## 9. Reviewed-manifest validation

The notebook accepts the original AUDIT bundle and upgrades it deterministically with a `reviewed_label` column. Ten locked human-confirmed Tier-B rows are relabeled; the two exact duplicate conflicts remain dropped. All other candidates remain `KEEP` or `AMBIGUOUS_KEEP`.


In [5]:
MANIFEST_HASH = None
REVIEW_MANIFEST_VALIDATION = {'required': False, 'run_mode': RUN_MODE}
OBJECTIVE_DROP_INDICES = np.array([], dtype=np.int64)
SEMANTIC_DROP_INDICES = np.array([], dtype=np.int64)
ALL_DROP_INDICES = np.array([], dtype=np.int64)
SEMANTIC_RELABEL_INDICES = np.array([], dtype=np.int64)
RELABEL_TARGET_BY_INDEX = {}
FINAL_REVIEW_MANIFEST = pd.DataFrame(columns=REVIEW_REQUIRED_COLUMNS)
AUDIT_RANKING = pd.DataFrame()
SCREEN_ROOT = None
REVIEW_VISUAL_FILES = []

def bounded_bundle_candidates(override, required):
    candidates = []
    if override:
        candidates.append(Path(override))
    candidates.append(ARTIFACT_DIR)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for marker in kaggle_input.rglob(required[0]):
            if marker.is_file():
                candidates.append(marker.parent)
    unique = []
    seen = set()
    for path in candidates:
        key = str(path)
        if key not in seen:
            unique.append(path)
            seen.add(key)
    return [path for path in unique if all((path / item).is_file() for item in required)]

def copy_audit_visuals(root, summary):
    names = ['cleaning_candidate_confidence_distributions.png'] + list(summary.get('contact_sheets', ()))
    names = list(dict.fromkeys(str(name) for name in names))
    require(all(Path(name).name == name and Path(name).suffix.lower() == '.png' for name in names), 'Unsafe audit-visual filename.')
    require(all((root / name).is_file() for name in names), 'AUDIT review visuals are incomplete.')
    for name in names:
        source, destination = root / name, ARTIFACT_DIR / name
        if source.resolve() != destination.resolve():
            shutil.copy2(source, destination)
    return names

def apply_locked_review_decisions(review):
    review = review.copy()
    if 'reviewed_label' not in review.columns:
        review.insert(review.columns.get_loc('original_label') + 1, 'reviewed_label', pd.to_numeric(review['original_label'], errors='raise').astype(int))
    review['reviewed_label'] = pd.to_numeric(review['reviewed_label'], errors='raise').astype(int)
    review['review_action'] = review['review_action'].astype(str)
    for relative_path, decision in LOCKED_RELABEL_DECISIONS.items():
        mask = review['relative_path'].astype(str) == relative_path
        require(int(mask.sum()) == 1, 'Locked relabel row absent or duplicated: {}.'.format(relative_path))
        index = review.index[mask][0]
        require(int(review.at[index, 'original_label']) == int(decision['original_label']), 'Locked relabel original label mismatch: {}.'.format(relative_path))
        require(int(review.at[index, 'oof_predicted_label']) == int(decision['reviewed_label']), 'Locked relabel target is not supported by the cross-fitted OOF candidate: {}.'.format(relative_path))
        require(str(review.at[index, 'candidate_tier']) == TIER_B, 'Locked relabel must be Tier B: {}.'.format(relative_path))
        review.at[index, 'reviewed_label'] = int(decision['reviewed_label'])
        review.at[index, 'review_action'] = 'RELABEL_OBVIOUS_MISLABEL'
        review.at[index, 'review_reason'] = decision['reason']
        review.at[index, 'reviewer'] = LOCKED_REVIEWER
        review.at[index, 'review_timestamp'] = LOCKED_REVIEW_TIMESTAMP
    for relative_path in LOCKED_OBJECTIVE_DROP_PATHS:
        mask = review['relative_path'].astype(str) == relative_path
        require(int(mask.sum()) == 1, 'Locked objective-drop row absent or duplicated: {}.'.format(relative_path))
        index = review.index[mask][0]
        require(str(review.at[index, 'candidate_tier']) == TIER_A, 'Locked objective drop must be Tier A: {}.'.format(relative_path))
        require(str(review.at[index, 'review_action']) == 'DROP_CORRUPT_OR_UNUSABLE', 'Objective duplicate conflict was changed: {}.'.format(relative_path))
        review.at[index, 'reviewed_label'] = int(review.at[index, 'original_label'])
    locked_paths = set(LOCKED_RELABEL_DECISIONS)
    other_tier_b = (review['candidate_tier'].astype(str) == TIER_B) & (~review['relative_path'].astype(str).isin(locked_paths))
    require(review.loc[other_tier_b, 'review_action'].astype(str).isin(('KEEP', 'AMBIGUOUS_KEEP')).all(), 'Unregistered Tier-B decisions are forbidden.')
    review.loc[review['review_action'].isin(('KEEP', 'AMBIGUOUS_KEEP', 'DROP_OBVIOUS_MISLABEL', 'DROP_CORRUPT_OR_UNUSABLE')), 'reviewed_label'] = review.loc[review['review_action'].isin(('KEEP', 'AMBIGUOUS_KEEP', 'DROP_OBVIOUS_MISLABEL', 'DROP_CORRUPT_OR_UNUSABLE')), 'original_label'].astype(int)
    return review

def validate_review_manifest(review, ranking, summary):
    review = apply_locked_review_decisions(review)
    require(set(review.columns) == set(REVIEW_REQUIRED_COLUMNS), 'Reviewed manifest columns must match the V9-v2 review contract exactly.')
    review = review[list(REVIEW_REQUIRED_COLUMNS)].copy().sort_values('sample_index').reset_index(drop=True)
    ranking = ranking.sort_values('sample_index').reset_index(drop=True)
    require(len(review) == len(ranking) and review['sample_index'].is_unique, 'Reviewed manifest row set/uniqueness failed.')
    require(review['sample_index'].astype(int).tolist() == ranking['sample_index'].astype(int).tolist(), 'Reviewed sample indices differ from AUDIT ranking.')
    immutable = [column for column in AUDIT_CONTRACT_COLUMNS if column not in {'review_action', 'review_reason', 'reviewer', 'review_timestamp'}]
    for column in immutable:
        if column in {'oof_original_probability', 'oof_top_probability', 'oof_margin'}:
            require(np.allclose(pd.to_numeric(review[column]), pd.to_numeric(ranking[column]), atol=1e-12, rtol=0.0), 'Immutable float field changed: {}.'.format(column))
        else:
            require(review[column].astype(str).tolist() == ranking[column].astype(str).tolist(), 'Immutable field changed: {}.'.format(column))
    computed_contract = candidate_contract_sha256(ranking)
    require(computed_contract == summary.get('candidate_contract_sha256'), 'AUDIT candidate contract hash mismatch.')
    require({'file_sha256', 'exact_duplicate_conflict'}.issubset(ranking.columns), 'Objective duplicate evidence is missing from the AUDIT ranking.')
    require(set(review['review_action'].astype(str)).issubset(ALLOWED_ACTIONS), 'Unknown review action.')
    review['reviewed_label'] = pd.to_numeric(review['reviewed_label'], errors='raise').astype(int)
    require(review['reviewed_label'].isin(range(N_CLASSES)).all(), 'reviewed_label must be one of 0,1,2.')
    mapping = fold_assignments.reset_index().rename(columns={'index': 'sample_index', 'image_id': 'relative_path', 'label': 'original_label'})[['sample_index', 'relative_path', 'fold', 'original_label']]
    checked = review.merge(mapping, on='sample_index', suffixes=('', '_canonical'), validate='one_to_one')
    for column in ('relative_path', 'fold', 'original_label'):
        require(checked[column].astype(str).tolist() == checked[column + '_canonical'].astype(str).tolist(), 'Reviewed mapping changed: {}.'.format(column))
    require(not review['relative_path'].astype(str).map(contains_forbidden_test_reference).any(), 'Forbidden path in reviewed manifest.')
    treatment_mask = review['review_action'].isin(('DROP_OBVIOUS_MISLABEL', 'DROP_CORRUPT_OR_UNUSABLE', 'RELABEL_OBVIOUS_MISLABEL'))
    require(review.loc[treatment_mask, 'review_reason'].astype(str).str.strip().ne('').all(), 'DROP/RELABEL rows require reasons.')
    require(review.loc[treatment_mask, 'reviewer'].astype(str).str.strip().ne('').all(), 'DROP/RELABEL rows require reviewer identity.')
    require(pd.to_datetime(review.loc[treatment_mask, 'review_timestamp'], utc=True, errors='coerce').notna().all(), 'DROP/RELABEL rows require parseable timestamps.')
    semantic_drop = review['review_action'] == 'DROP_OBVIOUS_MISLABEL'
    semantic_relabel = review['review_action'] == 'RELABEL_OBVIOUS_MISLABEL'
    objective = review['review_action'] == 'DROP_CORRUPT_OR_UNUSABLE'
    require((review.loc[semantic_drop | semantic_relabel, 'candidate_tier'] == TIER_B).all(), 'Semantic treatment requires Tier B.')
    require((review.loc[semantic_drop | semantic_relabel, 'selected_baseline_consensus'].astype(str).str.lower() == 'true').all(), 'Semantic treatment requires selected/baseline OOF consensus.')
    require((review.loc[semantic_relabel, 'reviewed_label'] != review.loc[semantic_relabel, 'original_label']).all(), 'RELABEL rows must change the label.')
    require((review.loc[semantic_relabel, 'reviewed_label'] == review.loc[semantic_relabel, 'oof_predicted_label'].astype(int)).all(), 'RELABEL targets must match the locked cross-fitted OOF candidate.')
    require((review.loc[~semantic_relabel, 'reviewed_label'] == review.loc[~semantic_relabel, 'original_label'].astype(int)).all(), 'Only RELABEL rows may change reviewed_label.')
    require((review.loc[objective, 'candidate_tier'] == TIER_A).all() and review.loc[objective, 'evidence_source'].astype(str).str.contains('RAW_TRAIN_OBJECTIVE_AUDIT').all(), 'Objective DROP requires Tier-A raw evidence.')
    duplicate_decisions = ranking[['sample_index', 'file_sha256', 'exact_duplicate_conflict']].merge(review[['sample_index', 'review_action']], on='sample_index', validate='one_to_one')
    duplicate_decisions = duplicate_decisions[duplicate_decisions['exact_duplicate_conflict'].astype(str).str.lower() == 'true']
    for _file_hash, group in duplicate_decisions.groupby('file_sha256'):
        require((group['review_action'] == 'DROP_CORRUPT_OR_UNUSABLE').all(), 'Every exact duplicate-conflict copy must be dropped.')
    manifest_drop_mask = semantic_drop | objective
    c1_drop_mask = semantic_drop | objective | semantic_relabel
    c2_drop_mask = semantic_drop | objective
    manifest_drop_count = int(manifest_drop_mask.sum())
    c1_drop_count = int(c1_drop_mask.sum())
    relabel_count = int(semantic_relabel.sum())
    require(c1_drop_count <= MAX_TOTAL_DROPS and relabel_count <= MAX_TOTAL_RELABELS, 'Treatment count exceeds locked maximum.')
    before = np.array([EXPECTED_CLASS_COUNTS[label] for label in range(N_CLASSES)], dtype=np.int64)
    c1_drops = np.bincount(review.loc[c1_drop_mask, 'original_label'].astype(int), minlength=N_CLASSES)
    c2_drops = np.bincount(review.loc[c2_drop_mask, 'original_label'].astype(int), minlength=N_CLASSES)
    c1_after = before - c1_drops
    relabel_out = np.bincount(review.loc[semantic_relabel, 'original_label'].astype(int), minlength=N_CLASSES)
    relabel_in = np.bincount(review.loc[semantic_relabel, 'reviewed_label'].astype(int), minlength=N_CLASSES)
    c2_after = before - c2_drops - relabel_out + relabel_in
    require((c1_drops / before <= MAX_DROP_FRACTION_PER_CLASS + 1e-12).all(), 'C1 per-class deletion fraction exceeds limit.')
    require(np.abs(c1_after / c1_after.sum() - before / before.sum()).max() <= MAX_CLASS_PROPORTION_SHIFT, 'C1 drop-only class-proportion shift exceeds limit.')
    require(np.abs(c2_after / c2_after.sum() - before / before.sum()).max() <= MAX_CLASS_PROPORTION_SHIFT, 'C2 relabel class-proportion shift exceeds limit.')
    action_counts = {str(key): int(value) for key, value in review['review_action'].value_counts().items()}
    relabel_matrix = review.loc[semantic_relabel].groupby(['original_label', 'reviewed_label']).size().rename('rows').reset_index().to_dict(orient='records')
    return review, {'candidate_contract_sha256': computed_contract, 'ranking_file_sha256': summary.get('ranking_file_sha256'), 'rows': len(review), 'manifest_drop_count': manifest_drop_count, 'c1_drop_count': c1_drop_count, 'objective_drops': int(objective.sum()), 'semantic_drops': int(semantic_drop.sum()), 'semantic_relabels': relabel_count, 'action_counts': action_counts, 'class_counts_before': before.tolist(), 'c1_drop_counts_by_class': c1_drops.tolist(), 'c1_counts_after': c1_after.tolist(), 'c2_objective_drop_counts_by_class': c2_drops.tolist(), 'c2_counts_after_relabel': c2_after.tolist(), 'relabel_matrix': relabel_matrix, 'pass': True}

if RUN_MODE == 'SCREEN':
    matches = bounded_bundle_candidates(os.environ.get('V9_AUDIT_BUNDLE_PATH', ''), ('cleaning_candidate_ranking.csv', 'cleaning_candidate_summary.json', 'cleaning_review_manifest.csv'))
    if len(matches) != 1:
        fail_closed('audit_bundle_discovery', 'SCREEN requires exactly one complete AUDIT bundle; found {}.'.format(len(matches)), {'matches': [str(path) for path in matches]})
    AUDIT_ROOT = matches[0]
    AUDIT_RANKING = pd.read_csv(AUDIT_ROOT / 'cleaning_candidate_ranking.csv')
    AUDIT_SUMMARY = read_json(AUDIT_ROOT / 'cleaning_candidate_summary.json')
    require(sha256_file(AUDIT_ROOT / 'cleaning_candidate_ranking.csv') == AUDIT_SUMMARY.get('ranking_file_sha256'), 'AUDIT ranking file hash mismatch.')
    REVIEW_VISUAL_FILES = copy_audit_visuals(AUDIT_ROOT, AUDIT_SUMMARY)
    reviewed = pd.read_csv(AUDIT_ROOT / 'cleaning_review_manifest.csv', keep_default_na=False)
    FINAL_REVIEW_MANIFEST, review_details = validate_review_manifest(reviewed, AUDIT_RANKING, AUDIT_SUMMARY)
    if (AUDIT_ROOT / 'cleaning_candidate_ranking.csv').resolve() != (ARTIFACT_DIR / 'cleaning_candidate_ranking.csv').resolve():
        shutil.copy2(AUDIT_ROOT / 'cleaning_candidate_ranking.csv', ARTIFACT_DIR / 'cleaning_candidate_ranking.csv')
    write_json(AUDIT_SUMMARY, ARTIFACT_DIR / 'cleaning_candidate_summary.json')
    write_csv(FINAL_REVIEW_MANIFEST, ARTIFACT_DIR / 'cleaning_review_manifest.csv')
    write_csv(FINAL_REVIEW_MANIFEST, ARTIFACT_DIR / 'cleaning_manifest_final.csv')
    MANIFEST_HASH = sha256_file(ARTIFACT_DIR / 'cleaning_manifest_final.csv')
    write_text(MANIFEST_HASH + '\n', ARTIFACT_DIR / 'cleaning_manifest_sha256.txt')
    REVIEW_MANIFEST_VALIDATION = {**provenance(MANIFEST_HASH), **review_details, 'audit_root': str(AUDIT_ROOT), 'locked_review_decisions_applied': True, 'immutable_review_fields_enforced': True, 'validation_rows_filtered': False, 'training_only_relabeling': True}
    write_json(REVIEW_MANIFEST_VALIDATION, ARTIFACT_DIR / 'review_manifest_validation.json')
elif RUN_MODE == 'FULL_CONFIRMATION':
    matches = bounded_bundle_candidates(os.environ.get('V9_SCREEN_BUNDLE_PATH', ''), ('locked_v9_contract.json', 'review_manifest_validation.json', 'cleaning_review_manifest.csv', 'cleaning_manifest_final.csv', 'cleaning_manifest_sha256.txt', 'cleaning_candidate_ranking.csv', 'cleaning_candidate_summary.json', 'control_reproduction.json', 'fold0_candidate_comparison.csv', 'fold0_candidate_metrics.json', 'fold0_decision.json'))
    if len(matches) != 1:
        fail_closed('screen_bundle_discovery', 'FULL_CONFIRMATION requires exactly one passed SCREEN bundle; found {}.'.format(len(matches)), {'matches': [str(path) for path in matches]})
    SCREEN_ROOT = matches[0]
    AUDIT_RANKING = pd.read_csv(SCREEN_ROOT / 'cleaning_candidate_ranking.csv')
    AUDIT_SUMMARY = read_json(SCREEN_ROOT / 'cleaning_candidate_summary.json')
    require(sha256_file(SCREEN_ROOT / 'cleaning_candidate_ranking.csv') == AUDIT_SUMMARY.get('ranking_file_sha256'), 'SCREEN ranking file hash mismatch.')
    REVIEW_VISUAL_FILES = copy_audit_visuals(SCREEN_ROOT, AUDIT_SUMMARY)
    reviewed = pd.read_csv(SCREEN_ROOT / 'cleaning_manifest_final.csv', keep_default_na=False)
    FINAL_REVIEW_MANIFEST, review_details = validate_review_manifest(reviewed, AUDIT_RANKING, AUDIT_SUMMARY)
    expected_hash = (SCREEN_ROOT / 'cleaning_manifest_sha256.txt').read_text(encoding='utf-8').strip()
    require(sha256_file(SCREEN_ROOT / 'cleaning_manifest_final.csv') == expected_hash, 'SCREEN cleaning-manifest hash changed.')
    screen_review_validation = read_json(SCREEN_ROOT / 'review_manifest_validation.json')
    require(screen_review_validation.get('pass') is True and screen_review_validation.get('cleaning_manifest_sha256') == expected_hash and screen_review_validation.get('candidate_contract_sha256') == review_details['candidate_contract_sha256'], 'SCREEN review-validation evidence changed.')
    MANIFEST_HASH = expected_hash
    for filename in ('cleaning_candidate_ranking.csv', 'cleaning_candidate_summary.json', 'cleaning_review_manifest.csv', 'cleaning_manifest_final.csv', 'cleaning_manifest_sha256.txt', 'review_manifest_validation.json', 'locked_v9_contract.json', 'control_reproduction.json', 'fold0_candidate_comparison.csv', 'fold0_candidate_metrics.json', 'fold0_paired_error_delta.csv', 'fold0_corrected_samples.csv', 'fold0_regressed_samples.csv', 'fold0_treatment_impact.csv', 'fold0_confusion_comparison.png', 'fold0_confidence_margin_distributions.png', 'fold0_diagnostic_summary.json', 'fold0_decision.json'):
        source = SCREEN_ROOT / filename
        if source.is_file() and source.resolve() != (ARTIFACT_DIR / filename).resolve():
            shutil.copy2(source, ARTIFACT_DIR / filename)
    source_heads = SCREEN_ROOT / 'heads'
    if source_heads.is_dir() and source_heads.resolve() != HEADS_DIR.resolve():
        for source in source_heads.iterdir():
            if source.is_file():
                shutil.copy2(source, HEADS_DIR / source.name)
    REVIEW_MANIFEST_VALIDATION = {**provenance(MANIFEST_HASH), **review_details, 'screen_root': str(SCREEN_ROOT), 'screen_hash_match': True, 'validation_rows_filtered': False, 'training_only_relabeling': True}
    write_json(REVIEW_MANIFEST_VALIDATION, ARTIFACT_DIR / 'review_manifest_validation.json')

if RUN_MODE != 'AUDIT':
    OBJECTIVE_DROP_INDICES = FINAL_REVIEW_MANIFEST.loc[FINAL_REVIEW_MANIFEST['review_action'] == 'DROP_CORRUPT_OR_UNUSABLE', 'sample_index'].astype(int).to_numpy()
    SEMANTIC_DROP_INDICES = FINAL_REVIEW_MANIFEST.loc[FINAL_REVIEW_MANIFEST['review_action'] == 'RELABEL_OBVIOUS_MISLABEL', 'sample_index'].astype(int).to_numpy()
    ALL_DROP_INDICES = np.unique(np.concatenate((OBJECTIVE_DROP_INDICES, SEMANTIC_DROP_INDICES))).astype(np.int64)
    SEMANTIC_RELABEL_INDICES = FINAL_REVIEW_MANIFEST.loc[FINAL_REVIEW_MANIFEST['review_action'] == 'RELABEL_OBVIOUS_MISLABEL', 'sample_index'].astype(int).to_numpy()
    RELABEL_TARGET_BY_INDEX = {int(row.sample_index): int(row.reviewed_label) for row in FINAL_REVIEW_MANIFEST[FINAL_REVIEW_MANIFEST['review_action'] == 'RELABEL_OBVIOUS_MISLABEL'].itertuples(index=False)}
    require(len(OBJECTIVE_DROP_INDICES) == 2 and len(SEMANTIC_RELABEL_INDICES) == 10, 'Locked treatment cardinality must be exactly 2 objective drops and 10 relabels.')
    print('Validated manifest:', MANIFEST_HASH, '| objective drops:', len(OBJECTIVE_DROP_INDICES), '| semantic rows:', len(SEMANTIC_RELABEL_INDICES))


RuntimeError: DROP/RELABEL rows require parseable timestamps.

## 10. Pre-registered candidates and gates

- C0: exact V7 control.
- C1: objective drops + semantic drop-only.
- C2: objective drops + semantic relabel.
- C3: fixed `0.50*C0 + 0.50*C2` probability blend.

Only training rows are treated. Validation stays complete and original.


In [ ]:
class CachedRows(Dataset):
    def __init__(self, indices, label_overrides=None):
        self.indices = [int(index) for index in indices]
        self.label_overrides = {int(key): int(value) for key, value in (label_overrides or {}).items()}
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, position):
        index = self.indices[position]
        label = self.label_overrides.get(index, int(cached_labels[index].item()))
        return cached_final_features[index], torch.tensor(label, dtype=torch.long), index

class RegularizedHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.LayerNorm(2 * HIDDEN_SIZE), nn.Linear(2 * HIDDEN_SIZE, HEAD_HIDDEN), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(HEAD_HIDDEN, N_CLASSES))
    def forward(self, features):
        return self.network(torch.cat((features[:, 0], features[:, 1]), dim=1))

def predict_logits(model, indices):
    indices = [int(index) for index in indices]
    loader = DataLoader(CachedRows(indices), batch_size=PREDICT_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)
    parts, observed = [], []
    model.eval()
    with torch.inference_mode():
        for features, _labels, batch_indices in loader:
            require(torch.isfinite(features).all().item(), 'Non-finite prediction features.')
            logits = model(features.to(DEVICE, dtype=torch.float32, non_blocking=PIN_MEMORY))
            require(torch.isfinite(logits).all().item(), 'Non-finite logits.')
            parts.append(logits.cpu())
            observed.extend(int(index) for index in batch_indices.tolist())
    require(observed == indices, 'Prediction order changed.')
    return torch.cat(parts).numpy().astype(np.float32)

def report_frame(report, metadata):
    rows = []
    for label in list(CLASS_NAMES) + ['macro avg', 'weighted avg']:
        rows.append({'label': label, **report[label], **metadata})
    return pd.DataFrame(rows)

def prediction_frame(indices, probabilities, candidate, stage):
    indices = np.asarray(indices, dtype=np.int64)
    probabilities = normalize_probabilities(probabilities)
    labels = cached_labels[torch.from_numpy(indices)].numpy()
    predictions = probabilities.argmax(axis=1)
    top = np.sort(probabilities, axis=1)[:, -2:]
    return pd.DataFrame({'sample_index': indices, 'relative_path': fold_assignments.iloc[indices]['image_id'].astype(str).to_numpy(), 'label': labels, 'prediction': predictions, 'prob_0': probabilities[:, 0], 'prob_1': probabilities[:, 1], 'prob_2': probabilities[:, 2], 'confidence': top[:, 1], 'margin': top[:, 1] - top[:, 0], 'correct': labels == predictions, 'fold': cached_folds[torch.from_numpy(indices)].numpy(), 'candidate': candidate, 'stage': stage, 'cleaning_manifest_sha256': MANIFEST_HASH})

def train_head(candidate, validation_fold, global_drop_indices, global_relabel_map, stage):
    require(RUN_MODE in {'SCREEN', 'FULL_CONFIRMATION'} and validation_fold in range(N_FOLDS), 'Training requested in invalid mode/fold.')
    fold_values = cached_folds.numpy()
    all_train = np.flatnonzero(fold_values != validation_fold)
    valid_indices = np.flatnonzero(fold_values == validation_fold)
    active_drops = np.intersect1d(all_train, np.asarray(global_drop_indices, dtype=np.int64), assume_unique=False)
    train_indices = np.setdiff1d(all_train, active_drops, assume_unique=False)
    active_relabel_map = {int(index): int(target) for index, target in global_relabel_map.items() if int(index) in set(all_train.tolist()) and int(index) not in set(active_drops.tolist())}
    require(not np.intersect1d(valid_indices, active_drops).size and not set(valid_indices.tolist()).intersection(active_relabel_map), 'Validation was filtered or relabeled.')
    original_labels = cached_labels.numpy()
    training_labels = original_labels.copy()
    for index, target in active_relabel_map.items():
        require(target in range(N_CLASSES) and target != int(original_labels[index]), 'Invalid active relabel mapping.')
        training_labels[index] = target
    class_counts = np.bincount(training_labels[train_indices], minlength=N_CLASSES)
    require((class_counts > 0).all(), 'Retained training partition misses a class.')
    class_weights = torch.tensor(class_counts.sum() / (N_CLASSES * class_counts), dtype=torch.float32, device=DEVICE)
    seed_everything(SEED)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(CachedRows(train_indices, active_relabel_map), batch_size=TRAIN_BATCH_SIZE, shuffle=True, generator=generator, num_workers=0, pin_memory=PIN_MEMORY, drop_last=False)
    model = RegularizedHead().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    best_state, best_epoch, best_f1, best_loss, stale = None, None, -1.0, float('inf'), 0
    history = []
    started = time.perf_counter()
    valid_labels = original_labels[valid_indices]
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        seen = 0
        max_grad = 0.0
        learning_rate = optimizer.param_groups[0]['lr']
        epoch_started = time.perf_counter()
        for features, batch_labels, _indices in loader:
            require(torch.isfinite(features).all().item(), 'Non-finite training features.')
            features = features.to(DEVICE, dtype=torch.float32, non_blocking=PIN_MEMORY)
            batch_labels = batch_labels.to(DEVICE, non_blocking=PIN_MEMORY)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            require(torch.isfinite(logits).all().item(), 'Non-finite training logits.')
            loss = criterion(logits, batch_labels)
            require(torch.isfinite(loss).item(), 'Non-finite loss.')
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            require(torch.isfinite(grad_norm).item(), 'Non-finite gradients.')
            optimizer.step()
            require(all(torch.isfinite(parameter).all().item() for parameter in model.parameters()), 'Non-finite parameters.')
            loss_sum += loss.item() * len(batch_labels)
            seen += len(batch_labels)
            max_grad = max(max_grad, float(grad_norm.item()))
        scheduler.step()
        probabilities = probabilities_from_logits(predict_logits(model, valid_indices))
        metrics = metric_bundle(valid_labels, probabilities)
        history.append({'candidate': candidate, 'stage': stage, 'validation_fold': validation_fold, 'epoch': epoch, 'train_loss': loss_sum / seen, 'valid_macro_f1': metrics['macro_f1'], 'valid_log_loss': metrics['multiclass_log_loss'], 'learning_rate': learning_rate, 'max_preclip_gradient_norm': max_grad, 'epoch_seconds': time.perf_counter() - epoch_started, 'seed': SEED, 'cleaning_manifest_sha256': MANIFEST_HASH})
        improved = metrics['macro_f1'] > best_f1 + 1e-8 or (abs(metrics['macro_f1'] - best_f1) <= 1e-8 and metrics['multiclass_log_loss'] < best_loss)
        if improved:
            best_f1, best_loss, best_epoch = metrics['macro_f1'], metrics['multiclass_log_loss'], epoch
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    require(best_state is not None, 'No best checkpoint selected.')
    model.load_state_dict(best_state, strict=True)
    probabilities = probabilities_from_logits(predict_logits(model, valid_indices))
    metrics = metric_bundle(valid_labels, probabilities)
    relabel_records = [{'sample_index': int(index), 'original_label': int(original_labels[index]), 'reviewed_label': int(target)} for index, target in sorted(active_relabel_map.items())]
    metrics.update({**provenance(MANIFEST_HASH), 'candidate': candidate, 'stage': stage, 'validation_fold': validation_fold, 'training_folds': sorted(set(range(N_FOLDS)) - {validation_fold}), 'best_epoch': best_epoch, 'parameter_count': sum(parameter.numel() for parameter in model.parameters()), 'head_architecture': 'LayerNorm(8192)-Linear(512)-GELU-Dropout(0.40)-Linear(3)', 'hyperparameters': {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'label_smoothing': LABEL_SMOOTHING, 'gradient_clip': GRAD_CLIP, 'batch_size': TRAIN_BATCH_SIZE}, 'train_class_counts_after_treatment': class_counts.tolist(), 'class_weights': class_weights.detach().cpu().tolist(), 'global_drop_indices': np.asarray(global_drop_indices, dtype=np.int64).tolist(), 'active_training_drop_indices': active_drops.tolist(), 'active_training_drop_count': len(active_drops), 'global_relabel_map': {str(key): int(value) for key, value in sorted(global_relabel_map.items())}, 'active_training_relabels': relabel_records, 'active_training_relabel_count': len(active_relabel_map), 'validation_rows_complete': True, 'validation_rows_filtered': False, 'validation_labels_original': True, 'training_seconds': time.perf_counter() - started})
    stem = '{}_{}_fold{}_seed{}'.format(stage, candidate, validation_fold, SEED)
    metadata = {**provenance(MANIFEST_HASH), 'candidate': candidate, 'stage': stage, 'validation_fold': validation_fold}
    write_csv(pd.DataFrame(history), HEADS_DIR / (stem + '_history.csv'))
    write_csv(report_frame(metrics['classification_report'], metadata), HEADS_DIR / (stem + '_classification_report.csv'))
    predictions = prediction_frame(valid_indices, probabilities, candidate, stage)
    write_csv(predictions, HEADS_DIR / (stem + '_predictions.csv'))
    checkpoint_metadata = {**provenance(MANIFEST_HASH), **metadata, 'architecture': 'LayerNorm(8192)-Linear(512)-GELU-Dropout(0.40)-Linear(3)', 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'label_smoothing': LABEL_SMOOTHING, 'gradient_clip': GRAD_CLIP, 'global_drop_indices': np.asarray(global_drop_indices, dtype=np.int64).tolist(), 'active_training_drop_indices': active_drops.tolist(), 'global_relabel_map': {str(key): int(value) for key, value in sorted(global_relabel_map.items())}, 'active_training_relabels': relabel_records, 'checkpoint_selection': 'validation Macro F1 then lower log loss'}
    checkpoint_path = HEADS_DIR / (stem + '.pt')
    atomic_torch_save({'metadata': checkpoint_metadata, 'best_epoch': best_epoch, 'state_dict': best_state}, checkpoint_path)
    persisted = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    require(persisted.get('metadata') == checkpoint_metadata and persisted.get('best_epoch') == best_epoch, 'Checkpoint persistence failed.')
    metrics.update({'history_file': str(HEADS_DIR / (stem + '_history.csv')), 'report_file': str(HEADS_DIR / (stem + '_classification_report.csv')), 'prediction_file': str(HEADS_DIR / (stem + '_predictions.csv')), 'checkpoint_file': str(checkpoint_path)})
    write_json(metrics, HEADS_DIR / (stem + '_metrics.json'))
    result = {'indices': valid_indices, 'probabilities': probabilities, 'metrics': metrics, 'predictions': predictions, 'checkpoint_path': checkpoint_path, 'history': history}
    del persisted, model, loader, criterion, optimizer, scheduler, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

def persist_blend(control, cleaned, validation_fold, stage):
    require(np.array_equal(control['indices'], cleaned['indices']), 'Blend rows misaligned.')
    probabilities = normalize_probabilities(BLEND_WEIGHT_C0 * control['probabilities'] + BLEND_WEIGHT_C2 * cleaned['probabilities'])
    indices = control['indices']
    metrics = metric_bundle(cached_labels[torch.from_numpy(indices)].numpy(), probabilities)
    metrics.update({**provenance(MANIFEST_HASH), 'candidate': 'C3', 'stage': stage, 'validation_fold': validation_fold, 'head_architecture': 'fixed 0.50*C0 + 0.50*C2 probability blend', 'blend': {'C0': BLEND_WEIGHT_C0, 'C2': BLEND_WEIGHT_C2}, 'global_drop_indices': OBJECTIVE_DROP_INDICES.tolist(), 'active_training_drop_indices': cleaned['metrics']['active_training_drop_indices'], 'active_training_drop_count': cleaned['metrics']['active_training_drop_count'], 'global_relabel_map': {str(key): int(value) for key, value in sorted(RELABEL_TARGET_BY_INDEX.items())}, 'active_training_relabels': cleaned['metrics']['active_training_relabels'], 'active_training_relabel_count': cleaned['metrics']['active_training_relabel_count'], 'validation_rows_complete': True, 'validation_rows_filtered': False, 'control_checkpoint': str(control['checkpoint_path']), 'cleaned_checkpoint': str(cleaned['checkpoint_path'])})
    stem = '{}_C3_fold{}_seed{}'.format(stage, validation_fold, SEED)
    predictions = prediction_frame(indices, probabilities, 'C3', stage)
    write_csv(predictions, HEADS_DIR / (stem + '_predictions.csv'))
    write_csv(report_frame(metrics['classification_report'], {'candidate': 'C3', 'stage': stage, 'validation_fold': validation_fold, 'seed': SEED, 'cleaning_manifest_sha256': MANIFEST_HASH}), HEADS_DIR / (stem + '_classification_report.csv'))
    write_csv(pd.DataFrame(cleaned['history']).assign(candidate='C3', blend='0.50*C0+0.50*C2'), HEADS_DIR / (stem + '_history.csv'))
    checkpoint_path = HEADS_DIR / (stem + '.pt')
    atomic_torch_save({'metadata': {**provenance(MANIFEST_HASH), 'candidate': 'C3', 'stage': stage, 'validation_fold': validation_fold, 'blend': {'C0': BLEND_WEIGHT_C0, 'C2': BLEND_WEIGHT_C2}, 'control_checkpoint': str(control['checkpoint_path']), 'cleaned_checkpoint': str(cleaned['checkpoint_path'])}}, checkpoint_path)
    metrics.update({'history_file': str(HEADS_DIR / (stem + '_history.csv')), 'report_file': str(HEADS_DIR / (stem + '_classification_report.csv')), 'prediction_file': str(HEADS_DIR / (stem + '_predictions.csv')), 'checkpoint_file': str(checkpoint_path)})
    write_json(metrics, HEADS_DIR / (stem + '_metrics.json'))
    return {'indices': indices, 'probabilities': probabilities, 'metrics': metrics, 'predictions': predictions, 'checkpoint_path': checkpoint_path, 'history': cleaned['history']}

def paired_counts(labels, control_probabilities, candidate_probabilities):
    control = normalize_probabilities(control_probabilities).argmax(axis=1)
    candidate = normalize_probabilities(candidate_probabilities).argmax(axis=1)
    labels = np.asarray(labels, dtype=np.int64)
    return {'corrected': int(((control != labels) & (candidate == labels)).sum()), 'regressed': int(((control == labels) & (candidate != labels)).sum()), 'net_corrections': int(((control != labels) & (candidate == labels)).sum() - ((control == labels) & (candidate != labels)).sum()), 'disagreements': int((control != candidate).sum())}

def paired_diagnostics(prefix, control, candidate):
    indices = np.asarray(control['indices'], dtype=np.int64)
    require(np.array_equal(indices, candidate['indices']), 'Paired diagnostic rows misaligned.')
    labels = cached_labels[torch.from_numpy(indices)].numpy()
    cp, vp = normalize_probabilities(control['probabilities']), normalize_probabilities(candidate['probabilities'])
    c, v = cp.argmax(axis=1), vp.argmax(axis=1)
    reviewed = np.isin(indices, FINAL_REVIEW_MANIFEST['sample_index'].astype(int).to_numpy())
    status = np.select(((c != labels) & (v == labels), (c == labels) & (v != labels), (c == labels) & (v == labels)), ('corrected', 'regressed', 'unchanged_correct'), default='unchanged_wrong')
    table = pd.DataFrame({'sample_index': indices, 'relative_path': fold_assignments.iloc[indices]['image_id'].astype(str).to_numpy(), 'fold': cached_folds[torch.from_numpy(indices)].numpy(), 'label': labels, 'control_prediction': c, 'candidate_prediction': v, 'control_confidence': cp.max(axis=1), 'candidate_confidence': vp.max(axis=1), 'control_margin': np.sort(cp, axis=1)[:, -1] - np.sort(cp, axis=1)[:, -2], 'candidate_margin': np.sort(vp, axis=1)[:, -1] - np.sort(vp, axis=1)[:, -2], 'reviewed_candidate_row': reviewed, 'status': status, 'candidate': candidate['metrics']['candidate'], 'cleaning_manifest_sha256': MANIFEST_HASH})
    write_csv(table, ARTIFACT_DIR / (prefix + '_paired_error_delta.csv'))
    write_csv(table[table['status'] == 'corrected'], ARTIFACT_DIR / (prefix + '_corrected_samples.csv'))
    write_csv(table[table['status'] == 'regressed'], ARTIFACT_DIR / (prefix + '_regressed_samples.csv'))
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for axis, probabilities, title in zip(axes, (cp, vp), ('C0', candidate['metrics']['candidate'])):
        matrix = confusion_matrix(labels, probabilities.argmax(axis=1), labels=list(range(N_CLASSES)))
        axis.imshow(matrix, cmap='Blues')
        axis.set_title(title)
        axis.set_xticks(range(N_CLASSES), CLASS_NAMES, rotation=25)
        axis.set_yticks(range(N_CLASSES), CLASS_NAMES)
        for row in range(N_CLASSES):
            for column in range(N_CLASSES):
                axis.text(column, row, str(matrix[row, column]), ha='center', va='center')
    figure.tight_layout()
    figure.savefig(ARTIFACT_DIR / (prefix + '_confusion_comparison.png'), dpi=160, bbox_inches='tight')
    plt.close(figure)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(cp.max(axis=1), bins=30, alpha=0.55, label='C0')
    axes[0].hist(vp.max(axis=1), bins=30, alpha=0.55, label=candidate['metrics']['candidate'])
    axes[0].set_title('Top-confidence distribution')
    axes[1].hist(np.sort(cp, axis=1)[:, -1] - np.sort(cp, axis=1)[:, -2], bins=30, alpha=0.55, label='C0')
    axes[1].hist(np.sort(vp, axis=1)[:, -1] - np.sort(vp, axis=1)[:, -2], bins=30, alpha=0.55, label=candidate['metrics']['candidate'])
    axes[1].set_title('Top1-top2 margin distribution')
    for axis in axes:
        axis.legend()
    figure.tight_layout()
    figure.savefig(ARTIFACT_DIR / (prefix + '_confidence_margin_distributions.png'), dpi=160, bbox_inches='tight')
    plt.close(figure)

    def subset_metrics(mask, probabilities):
        if not np.asarray(mask, dtype=bool).any():
            return {'rows': 0}
        metrics = metric_bundle(labels[mask], probabilities[mask])
        return {key: metrics[key] for key in ('rows', 'macro_f1', 'accuracy', 'multiclass_log_loss', 'classification_report', 'recyclable_to_organic', 'organic_to_recyclable', 'total_errors')}

    def distribution(values):
        values = np.asarray(values, dtype=np.float64)
        return {'mean': float(values.mean()), 'median': float(np.median(values)), 'q10': float(np.quantile(values, 0.10)), 'q90': float(np.quantile(values, 0.90))}

    counts = paired_counts(labels, cp, vp)
    flat_correlation = float(np.corrcoef(cp.ravel(), vp.ravel())[0, 1]) if np.std(cp) > 0 and np.std(vp) > 0 else float(np.allclose(cp, vp))
    counts.update({'corrected_reviewed': int(((status == 'corrected') & reviewed).sum()), 'corrected_nonreviewed': int(((status == 'corrected') & ~reviewed).sum()), 'regressed_reviewed': int(((status == 'regressed') & reviewed).sum()), 'regressed_nonreviewed': int(((status == 'regressed') & ~reviewed).sum()), 'probability_correlation_flat': flat_correlation, 'disagreement_rate': float((c != v).mean()), 'validation_rows': len(indices), 'reviewed_validation_rows': int(reviewed.sum()), 'status_counts': {str(key): int(value) for key, value in pd.Series(status).value_counts().items()}, 'reviewed_vs_nonreviewed_metrics': {'reviewed': {'C0': subset_metrics(reviewed, cp), 'candidate': subset_metrics(reviewed, vp)}, 'nonreviewed': {'C0': subset_metrics(~reviewed, cp), 'candidate': subset_metrics(~reviewed, vp)}}, 'confidence_margin_summary': {'C0': {'confidence': distribution(cp.max(axis=1)), 'margin': distribution(np.sort(cp, axis=1)[:, -1] - np.sort(cp, axis=1)[:, -2])}, 'candidate': {'confidence': distribution(vp.max(axis=1)), 'margin': distribution(np.sort(vp, axis=1)[:, -1] - np.sort(vp, axis=1)[:, -2])}}})
    write_json({**provenance(MANIFEST_HASH), 'candidate': candidate['metrics']['candidate'], **counts}, ARTIFACT_DIR / (prefix + '_diagnostic_summary.json'))
    return table, counts

def treatment_impact(candidate, validation_fold):
    drops = np.asarray(candidate['metrics'].get('active_training_drop_indices', []), dtype=np.int64)
    relabels = candidate['metrics'].get('active_training_relabels', [])
    rows = []
    original = cached_labels.numpy()
    for label in range(N_CLASSES):
        before = int(((cached_folds.numpy() != validation_fold) & (original == label)).sum())
        dropped = int((original[drops] == label).sum()) if len(drops) else 0
        relabeled_out = sum(int(row['original_label']) == label for row in relabels)
        relabeled_in = sum(int(row['reviewed_label']) == label for row in relabels)
        after = before - dropped - relabeled_out + relabeled_in
        globally_flagged_validation_rows_retained = int(sum(int(cached_folds[int(index)].item()) == validation_fold for index in set(candidate['metrics'].get('global_drop_indices', [])) | {int(key) for key in candidate['metrics'].get('global_relabel_map', {})}))
        rows.append({'candidate': candidate['metrics']['candidate'], 'validation_fold': validation_fold, 'class_label': label, 'class_name': CLASS_MAPPING[label], 'training_rows_before': before, 'dropped': dropped, 'relabeled_out': relabeled_out, 'relabeled_in': relabeled_in, 'training_rows_after_treatment': after, 'globally_flagged_validation_rows_retained': globally_flagged_validation_rows_retained, 'validation_rows_filtered': False, 'validation_labels_modified': False, 'cleaning_manifest_sha256': MANIFEST_HASH})
    return rows

LOCKED_V9_CONTRACT = None
LOCKED_V9_CONTRACT_SIGNATURE = None
DROP_SETS = {'C0': np.array([], dtype=np.int64), 'C1': ALL_DROP_INDICES, 'C2': OBJECTIVE_DROP_INDICES, 'C3': OBJECTIVE_DROP_INDICES}
RELABEL_MAPS = {'C0': {}, 'C1': {}, 'C2': RELABEL_TARGET_BY_INDEX, 'C3': RELABEL_TARGET_BY_INDEX}
if RUN_MODE == 'SCREEN':
    enabled = ['C1', 'C2', 'C3'] if len(OBJECTIVE_DROP_INDICES) == 2 and len(SEMANTIC_RELABEL_INDICES) == 10 else []
    LOCKED_V9_CONTRACT = {**provenance(MANIFEST_HASH), 'decision_timing': 'before fold-0 V9 scoring', 'candidate_contract_sha256': REVIEW_MANIFEST_VALIDATION['candidate_contract_sha256'], 'ranking_file_sha256': REVIEW_MANIFEST_VALIDATION['ranking_file_sha256'], 'control': 'C0 exact V7 regularized_mlp_final', 'enabled_candidates': enabled, 'definitions': {'C1': 'drop 2 objective conflicts plus drop 10 human-confirmed semantic mislabels', 'C2': 'drop 2 objective conflicts plus relabel 10 human-confirmed semantic mislabels', 'C3': {'C0': BLEND_WEIGHT_C0, 'C2': BLEND_WEIGHT_C2}}, 'global_drop_indices': {name: values.tolist() for name, values in DROP_SETS.items()}, 'global_relabel_maps': {name: {str(key): int(value) for key, value in sorted(values.items())} for name, values in RELABEL_MAPS.items()}, 'head': {'architecture': 'LayerNorm(8192)-Linear(512)-GELU-Dropout(0.40)-Linear(3)', 'weighted_cross_entropy': True, 'label_smoothing': LABEL_SMOOTHING, 'optimizer': 'AdamW', 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'scheduler': 'CosineAnnealingLR', 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'gradient_clip': GRAD_CLIP}, 'fold0_gates': FOLD0_GATES, 'full_gates': FULL_GATES, 'validation_complete': True, 'validation_labels_original': True, 'training_only_relabeling': True}
    LOCKED_V9_CONTRACT_SIGNATURE = sha256_json(LOCKED_V9_CONTRACT)
    LOCKED_V9_CONTRACT['contract_signature'] = LOCKED_V9_CONTRACT_SIGNATURE
    write_json(LOCKED_V9_CONTRACT, ARTIFACT_DIR / 'locked_v9_contract.json')
elif RUN_MODE == 'FULL_CONFIRMATION':
    LOCKED_V9_CONTRACT = read_json(SCREEN_ROOT / 'locked_v9_contract.json')
    LOCKED_V9_CONTRACT_SIGNATURE = LOCKED_V9_CONTRACT.pop('contract_signature')
    require(sha256_json(LOCKED_V9_CONTRACT) == LOCKED_V9_CONTRACT_SIGNATURE and LOCKED_V9_CONTRACT.get('cleaning_manifest_sha256') == MANIFEST_HASH, 'Locked SCREEN contract changed.')
    expected_drops = {name: values.tolist() for name, values in DROP_SETS.items()}
    expected_relabels = {name: {str(key): int(value) for key, value in sorted(values.items())} for name, values in RELABEL_MAPS.items()}
    require(set(LOCKED_V9_CONTRACT.get('enabled_candidates', ())).issubset(CANDIDATE_NAMES) and LOCKED_V9_CONTRACT.get('global_drop_indices') == expected_drops and LOCKED_V9_CONTRACT.get('global_relabel_maps') == expected_relabels and LOCKED_V9_CONTRACT.get('candidate_contract_sha256') == REVIEW_MANIFEST_VALIDATION['candidate_contract_sha256'] and LOCKED_V9_CONTRACT.get('ranking_file_sha256') == REVIEW_MANIFEST_VALIDATION['ranking_file_sha256'], 'Locked candidate/treatment definitions differ from the reviewed manifest.')
    LOCKED_V9_CONTRACT['contract_signature'] = LOCKED_V9_CONTRACT_SIGNATURE
    write_json(LOCKED_V9_CONTRACT, ARTIFACT_DIR / 'locked_v9_contract.json')


## 11. Exact V7 fold-0 control reproduction

SCREEN retrains C0 before any cleaning candidate and requires Macro-F1 tolerance 0.0002 plus exact predictions, error directions, and Electronic F1. FULL_CONFIRMATION loads the already-passed SCREEN fold-0 outputs rather than retraining them.

In [ ]:
C0_FOLD0 = None
CONTROL_REPRODUCTION = None
def result_from_files(root, stem):
    head_root = root / 'heads'
    paths = {suffix: head_root / (stem + suffix) for suffix in ('_predictions.csv', '_metrics.json', '_history.csv', '_classification_report.csv', '.pt')}
    require(all(path.is_file() for path in paths.values()), 'Incomplete persisted head bundle: {}.'.format(stem))
    predictions = pd.read_csv(paths['_predictions.csv']).sort_values('sample_index').reset_index(drop=True)
    metrics = read_json(paths['_metrics.json'])
    require({'sample_index', 'label', 'prediction', 'prob_0', 'prob_1', 'prob_2', 'fold', 'candidate', 'cleaning_manifest_sha256'}.issubset(predictions.columns) and len(predictions) > 0 and predictions['sample_index'].is_unique, 'Persisted prediction schema/rows invalid: {}.'.format(stem))
    require(metrics.get('cleaning_manifest_sha256') == MANIFEST_HASH and set(predictions['cleaning_manifest_sha256'].astype(str)) == {MANIFEST_HASH}, 'Persisted head manifest provenance mismatch: {}.'.format(stem))
    indices = predictions['sample_index'].astype(int).to_numpy()
    expected_candidate = stem.split('_')[1]
    require(metrics.get('candidate') == expected_candidate and set(predictions['candidate'].astype(str)) == {expected_candidate}, 'Persisted candidate identity mismatch: {}.'.format(stem))
    require(predictions['label'].astype(int).tolist() == cached_labels[torch.from_numpy(indices)].numpy().tolist() and predictions['fold'].astype(int).tolist() == cached_folds[torch.from_numpy(indices)].numpy().tolist(), 'Persisted labels/folds changed: {}.'.format(stem))
    probabilities = normalize_probabilities(predictions[['prob_0', 'prob_1', 'prob_2']].to_numpy())
    require(np.array_equal(probabilities.argmax(axis=1), predictions['prediction'].astype(int).to_numpy()), 'Persisted predictions disagree with probabilities: {}.'.format(stem))
    return {'indices': indices, 'probabilities': probabilities, 'metrics': metrics, 'predictions': predictions, 'checkpoint_path': paths['.pt'], 'history': pd.read_csv(paths['_history.csv']).to_dict(orient='records')}

if RUN_MODE == 'SCREEN':
    C0_FOLD0 = train_head('C0', DEVELOPMENT_FOLD, DROP_SETS['C0'], RELABEL_MAPS['C0'], 'screen')
    reference = selected_oof[selected_oof['fold'].astype(int) == DEVELOPMENT_FOLD].sort_values('row_index').reset_index(drop=True)
    require(C0_FOLD0['indices'].tolist() == reference['row_index'].astype(int).tolist(), 'C0/reference rows differ.')
    reproduced_predictions = C0_FOLD0['probabilities'].argmax(axis=1)
    reference_predictions = reference['prediction'].astype(int).to_numpy()
    macro_delta = abs(C0_FOLD0['metrics']['macro_f1'] - V7_FOLD0_CONTROL['macro_f1'])
    exact_predictions = np.array_equal(reproduced_predictions, reference_predictions)
    CONTROL_REPRODUCTION = {**provenance(MANIFEST_HASH), 'pass': macro_delta <= CONTROL_TOLERANCE and exact_predictions, 'macro_f1_delta': macro_delta, 'maximum_macro_f1_delta': CONTROL_TOLERANCE, 'exact_prediction_identity': exact_predictions, 'reproduced': C0_FOLD0['metrics'], 'reference': {'macro_f1': V7_FOLD0_CONTROL['macro_f1'], 'total_errors': V7_FOLD0_CONTROL['total_errors'], 'recyclable_to_organic': V7_FOLD0_CONTROL['recyclable_to_organic'], 'organic_to_recyclable': V7_FOLD0_CONTROL['organic_to_recyclable'], 'electronic_f1': V7_FOLD0_CONTROL['classification_report']['Electronic']['f1-score']}}
    CONTROL_REPRODUCTION['pass'] = CONTROL_REPRODUCTION['pass'] and C0_FOLD0['metrics']['total_errors'] == EXPECTED_V7_FOLD0_ERRORS and C0_FOLD0['metrics']['recyclable_to_organic'] == EXPECTED_V7_FOLD0_R2O and C0_FOLD0['metrics']['organic_to_recyclable'] == EXPECTED_V7_FOLD0_O2R and abs(C0_FOLD0['metrics']['classification_report']['Electronic']['f1-score'] - EXPECTED_V7_FOLD0_E_F1) <= 1e-12
    write_json(CONTROL_REPRODUCTION, ARTIFACT_DIR / 'control_reproduction.json')
    if not CONTROL_REPRODUCTION['pass']:
        fail_closed('control_reproduction', 'Exact V7 fold-0 control reproduction failed.', CONTROL_REPRODUCTION)
elif RUN_MODE == 'FULL_CONFIRMATION':
    C0_FOLD0 = result_from_files(SCREEN_ROOT, 'screen_C0_fold0_seed42')
    CONTROL_REPRODUCTION = read_json(SCREEN_ROOT / 'control_reproduction.json')
    require(CONTROL_REPRODUCTION.get('pass') is True and CONTROL_REPRODUCTION.get('cleaning_manifest_sha256') == MANIFEST_HASH, 'SCREEN control reproduction is invalid.')
if RUN_MODE != 'AUDIT':
    print('C0 fold-0 reproduction passed:', CONTROL_REPRODUCTION['pass'])


## 12. SCREEN fold-0 drop-vs-relabel candidates


In [ ]:
FOLD0_RESULTS = {}
if RUN_MODE == 'SCREEN':
    FOLD0_RESULTS['C0'] = C0_FOLD0
    for candidate in LOCKED_V9_CONTRACT['enabled_candidates']:
        if candidate in {'C1', 'C2'}:
            FOLD0_RESULTS[candidate] = train_head(candidate, DEVELOPMENT_FOLD, DROP_SETS[candidate], RELABEL_MAPS[candidate], 'screen')
        elif candidate == 'C3':
            require('C2' in FOLD0_RESULTS, 'C3 requires C2.')
            FOLD0_RESULTS['C3'] = persist_blend(C0_FOLD0, FOLD0_RESULTS['C2'], DEVELOPMENT_FOLD, 'screen')
    comparison = []
    metric_payload = {}
    treatment_rows = []
    for candidate, result in FOLD0_RESULTS.items():
        metrics = result['metrics']
        comparison.append({'candidate': candidate, 'macro_f1': metrics['macro_f1'], 'macro_f1_delta': metrics['macro_f1'] - C0_FOLD0['metrics']['macro_f1'], 'accuracy': metrics['accuracy'], 'log_loss': metrics['multiclass_log_loss'], 'recyclable_f1': metrics['classification_report']['Recyclable']['f1-score'], 'electronic_f1': metrics['classification_report']['Electronic']['f1-score'], 'organic_f1': metrics['classification_report']['Organic']['f1-score'], 'recyclable_to_organic': metrics['recyclable_to_organic'], 'organic_to_recyclable': metrics['organic_to_recyclable'], 'total_errors': metrics['total_errors'], 'active_training_drops': metrics.get('active_training_drop_count', 0), 'active_training_relabels': metrics.get('active_training_relabel_count', 0), 'cleaning_manifest_sha256': MANIFEST_HASH})
        metric_payload[candidate] = metrics
        treatment_rows.extend(treatment_impact(result, DEVELOPMENT_FOLD))
    write_csv(pd.DataFrame(comparison), ARTIFACT_DIR / 'fold0_candidate_comparison.csv')
    write_json({**provenance(MANIFEST_HASH), 'candidates': metric_payload}, ARTIFACT_DIR / 'fold0_candidate_metrics.json')
    write_csv(pd.DataFrame(treatment_rows), ARTIFACT_DIR / 'fold0_treatment_impact.csv')
    print(pd.DataFrame(comparison).to_string(index=False))


## 13. Fold-0 decision

Gates and ranking are applied exactly as pre-registered. Failure is a clean scientific rejection: no thresholds or manifest actions are changed and V7 remains final.

In [ ]:
FOLD0_DECISION = None
SELECTED_CANDIDATE = None
FOLD0_DIAGNOSTICS = None
if RUN_MODE == 'SCREEN':
    gate_results = {}
    labels = cached_labels[torch.from_numpy(C0_FOLD0['indices'])].numpy()
    control_metrics = C0_FOLD0['metrics']
    control_ro = control_metrics['recyclable_to_organic'] + control_metrics['organic_to_recyclable']
    for candidate in LOCKED_V9_CONTRACT['enabled_candidates']:
        result = FOLD0_RESULTS[candidate]
        metrics = result['metrics']
        paired = paired_counts(labels, C0_FOLD0['probabilities'], result['probabilities'])
        ro_errors = metrics['recyclable_to_organic'] + metrics['organic_to_recyclable']
        electronic_delta = metrics['classification_report']['Electronic']['f1-score'] - control_metrics['classification_report']['Electronic']['f1-score']
        conditions = {'macro_f1': metrics['macro_f1'] >= FOLD0_GATES['minimum_macro_f1'], 'gain': metrics['macro_f1'] - control_metrics['macro_f1'] >= FOLD0_GATES['minimum_gain'], 'total_errors': metrics['total_errors'] <= FOLD0_GATES['maximum_total_errors'], 'ro_error_reduction': control_ro - ro_errors >= FOLD0_GATES['minimum_ro_error_reduction'], 'electronic_preservation': electronic_delta >= -FOLD0_GATES['maximum_electronic_f1_degradation'], 'corrected': paired['corrected'] >= FOLD0_GATES['minimum_corrected'], 'net_corrections': paired['net_corrections'] >= FOLD0_GATES['minimum_net_corrections'], 'not_one_sample_dependent': paired['corrected'] >= 5 and paired['net_corrections'] >= 2, 'treatment_defensible': REVIEW_MANIFEST_VALIDATION['pass'], 'treatment_provenance_complete': (result['metrics'].get('active_training_drop_count', 0) + result['metrics'].get('active_training_relabel_count', 0)) > 0, 'validation_complete': result['metrics'].get('validation_rows_complete') is True, 'locked_contract_unchanged': sha256_json({key: value for key, value in LOCKED_V9_CONTRACT.items() if key != 'contract_signature'}) == LOCKED_V9_CONTRACT_SIGNATURE}
        gate_results[candidate] = {'pass': all(conditions.values()), 'conditions': conditions, 'observed': {'macro_f1': metrics['macro_f1'], 'gain': metrics['macro_f1'] - control_metrics['macro_f1'], 'total_errors': metrics['total_errors'], 'ro_errors': ro_errors, 'ro_error_reduction': control_ro - ro_errors, 'electronic_f1_delta': electronic_delta, **paired, 'active_training_drops': metrics.get('active_training_drop_count', 0), 'active_training_relabels': metrics.get('active_training_relabel_count', 0)}, 'gates': FOLD0_GATES}
    passing = [candidate for candidate in LOCKED_V9_CONTRACT['enabled_candidates'] if gate_results[candidate]['pass']]
    passing.sort(key=lambda candidate: (-FOLD0_RESULTS[candidate]['metrics']['macro_f1'], FOLD0_RESULTS[candidate]['metrics']['total_errors'], FOLD0_RESULTS[candidate]['metrics']['recyclable_to_organic'] + FOLD0_RESULTS[candidate]['metrics']['organic_to_recyclable'], -FOLD0_RESULTS[candidate]['metrics']['classification_report']['Electronic']['f1-score'], FOLD0_RESULTS[candidate]['metrics'].get('active_training_drop_count', 0) + FOLD0_RESULTS[candidate]['metrics'].get('active_training_relabel_count', 0), -paired_counts(labels, C0_FOLD0['probabilities'], FOLD0_RESULTS[candidate]['probabilities'])['disagreements']))
    SELECTED_CANDIDATE = passing[0] if passing else None
    diagnostic_candidate = SELECTED_CANDIDATE or (max(LOCKED_V9_CONTRACT['enabled_candidates'], key=lambda name: FOLD0_RESULTS[name]['metrics']['macro_f1']) if LOCKED_V9_CONTRACT['enabled_candidates'] else 'C0')
    _, FOLD0_DIAGNOSTICS = paired_diagnostics('fold0', C0_FOLD0, FOLD0_RESULTS[diagnostic_candidate])
    FOLD0_DECISION = {**provenance(MANIFEST_HASH), 'pass': SELECTED_CANDIDATE is not None, 'selected_candidate': SELECTED_CANDIDATE, 'diagnostic_candidate': diagnostic_candidate, 'gate_results': gate_results, 'candidate_order': LOCKED_V9_CONTRACT['enabled_candidates'], 'locked_contract_signature': LOCKED_V9_CONTRACT_SIGNATURE, 'reason': 'A locked audited-cleaning candidate passed every fold-0 gate.' if SELECTED_CANDIDATE else ('No complete locked drop-versus-relabel treatment exists.' if not LOCKED_V9_CONTRACT['enabled_candidates'] else 'No locked drop-versus-relabel candidate passed every fold-0 gate; retain V7.'), 'noise_interpretation': 'At least five C0 errors and two net errors were corrected; five-fold confirmation is still required.' if SELECTED_CANDIDATE else 'No robust fold-0 gain beyond the locked one-to-five-sample-noise guard was accepted.', 'confirmation_allowed': SELECTED_CANDIDATE is not None, 'manifest_frozen': True, 'validation_complete': True}
    write_json(FOLD0_DECISION, ARTIFACT_DIR / 'fold0_decision.json')
elif RUN_MODE == 'FULL_CONFIRMATION':
    FOLD0_DECISION = read_json(SCREEN_ROOT / 'fold0_decision.json')
    require(FOLD0_DECISION.get('pass') is True and FOLD0_DECISION.get('cleaning_manifest_sha256') == MANIFEST_HASH and FOLD0_DECISION.get('locked_contract_signature') == LOCKED_V9_CONTRACT_SIGNATURE, 'FULL_CONFIRMATION requires a hash-matched passed SCREEN decision.')
    SELECTED_CANDIDATE = FOLD0_DECISION['selected_candidate']
    require(SELECTED_CANDIDATE in LOCKED_V9_CONTRACT['enabled_candidates'] and FOLD0_DECISION.get('confirmation_allowed') is True, 'SCREEN-selected candidate is absent from the locked contract.')
if RUN_MODE != 'AUDIT':
    print('Fold-0 decision:', FOLD0_DECISION['reason'])


## 14. Conditional FULL_CONFIRMATION

Only the selected candidate from the passed fold-0 contract is run on folds 1–4. The same locked treatment manifest and label overrides are reused without retuning.


In [ ]:
FULL_GATE_RESULTS = None
CONFIRMATION_RESULTS = None
if RUN_MODE == 'FULL_CONFIRMATION':
    selected_stem = 'screen_{}_fold0_seed42'.format(SELECTED_CANDIDATE)
    selected_fold0 = result_from_files(SCREEN_ROOT, selected_stem)
    fold0_results = {'C0': C0_FOLD0, 'candidate': selected_fold0}
    if SELECTED_CANDIDATE == 'C3':
        fold0_results['cleaned'] = result_from_files(SCREEN_ROOT, 'screen_C2_fold0_seed42')
    fold_results = {0: fold0_results}
    per_fold_rows = []
    treatment_rows = []
    for fold in CONFIRMATION_FOLDS:
        control = train_head('C0', fold, DROP_SETS['C0'], RELABEL_MAPS['C0'], 'confirmation')
        if SELECTED_CANDIDATE == 'C3':
            cleaned = train_head('C2', fold, DROP_SETS['C2'], RELABEL_MAPS['C2'], 'confirmation')
            candidate = persist_blend(control, cleaned, fold, 'confirmation')
        else:
            candidate = train_head(SELECTED_CANDIDATE, fold, DROP_SETS[SELECTED_CANDIDATE], RELABEL_MAPS[SELECTED_CANDIDATE], 'confirmation')
        fold_results[fold] = {'C0': control, 'candidate': candidate}
        if SELECTED_CANDIDATE == 'C3':
            fold_results[fold]['cleaned'] = cleaned
    for fold in range(N_FOLDS):
        control, candidate = fold_results[fold]['C0'], fold_results[fold]['candidate']
        labels = cached_labels[torch.from_numpy(control['indices'])].numpy()
        paired = paired_counts(labels, control['probabilities'], candidate['probabilities'])
        cm, vm = control['metrics'], candidate['metrics']
        per_fold_rows.append({'fold': fold, 'candidate': SELECTED_CANDIDATE, 'control_macro_f1': cm['macro_f1'], 'candidate_macro_f1': vm['macro_f1'], 'macro_f1_delta': vm['macro_f1'] - cm['macro_f1'], 'control_total_errors': cm['total_errors'], 'candidate_total_errors': vm['total_errors'], 'control_ro_errors': cm['recyclable_to_organic'] + cm['organic_to_recyclable'], 'candidate_ro_errors': vm['recyclable_to_organic'] + vm['organic_to_recyclable'], 'control_electronic_f1': cm['classification_report']['Electronic']['f1-score'], 'candidate_electronic_f1': vm['classification_report']['Electronic']['f1-score'], **paired, 'candidate_beats_control': vm['macro_f1'] > cm['macro_f1'], 'cleaning_manifest_sha256': MANIFEST_HASH})
        treatment_rows.extend(treatment_impact(candidate, fold))
    write_csv(pd.DataFrame(per_fold_rows), ARTIFACT_DIR / 'confirmation_per_fold_comparison.csv')
    write_csv(pd.DataFrame(treatment_rows), ARTIFACT_DIR / 'confirmation_treatment_impact.csv')

    def concatenate_oof(role):
        indices = np.concatenate([fold_results[fold][role]['indices'] for fold in range(N_FOLDS)])
        probabilities = np.concatenate([fold_results[fold][role]['probabilities'] for fold in range(N_FOLDS)])
        order = np.argsort(indices)
        indices, probabilities = indices[order], probabilities[order]
        require(indices.tolist() == list(range(EXPECTED_ROWS)), '{} OOF coverage failed.'.format(role))
        return indices, normalize_probabilities(probabilities)

    oof_indices, control_probabilities = concatenate_oof('C0')
    candidate_indices, candidate_probabilities = concatenate_oof('candidate')
    require(np.array_equal(oof_indices, candidate_indices), 'Control/candidate OOF rows differ.')
    labels = cached_labels.numpy()
    control_metrics = metric_bundle(labels, control_probabilities)
    candidate_metrics = metric_bundle(labels, candidate_probabilities)
    control_exact = np.array_equal(control_probabilities.argmax(axis=1), selected_oof_probabilities.argmax(axis=1))
    require(abs(control_metrics['macro_f1'] - EXPECTED_V7_FIVEFOLD_MACRO_F1) <= CONTROL_TOLERANCE and control_exact, 'Five-fold C0 did not reproduce V7 OOF.')
    control_result = {'indices': oof_indices, 'probabilities': control_probabilities, 'metrics': {**control_metrics, 'candidate': 'C0'}}
    candidate_result = {'indices': candidate_indices, 'probabilities': candidate_probabilities, 'metrics': {**candidate_metrics, 'candidate': SELECTED_CANDIDATE}}
    paired_table, aggregate_paired = paired_diagnostics('confirmation', control_result, candidate_result)
    control_predictions = prediction_frame(oof_indices, control_probabilities, 'C0', 'confirmation_fivefold_oof')
    selected_predictions = prediction_frame(candidate_indices, candidate_probabilities, SELECTED_CANDIDATE, 'confirmation_fivefold_oof')
    write_csv(control_predictions, ARTIFACT_DIR / 'confirmation_control_oof_predictions.csv')
    write_csv(selected_predictions, ARTIFACT_DIR / 'confirmation_selected_oof_predictions.csv')
    comparison_rows = [{'role': 'reproduced_C0', 'candidate': 'C0', **{key: control_metrics[key] for key in ('macro_f1', 'accuracy', 'multiclass_log_loss', 'recyclable_to_organic', 'organic_to_recyclable', 'total_errors')}}, {'role': 'v9_candidate', 'candidate': SELECTED_CANDIDATE, **{key: candidate_metrics[key] for key in ('macro_f1', 'accuracy', 'multiclass_log_loss', 'recyclable_to_organic', 'organic_to_recyclable', 'total_errors')}}]
    cleaned_metrics = None
    if SELECTED_CANDIDATE == 'C3':
        cleaned_indices, cleaned_probabilities = concatenate_oof('cleaned')
        require(np.array_equal(cleaned_indices, candidate_indices), 'C2/C3 OOF rows differ.')
        cleaned_metrics = metric_bundle(labels, cleaned_probabilities)
        write_csv(prediction_frame(cleaned_indices, cleaned_probabilities, 'C2', 'confirmation_fivefold_oof'), ARTIFACT_DIR / 'confirmation_cleaned_oof_predictions.csv')
        comparison_rows.append({'role': 'cleaned_component', 'candidate': 'C2', **{key: cleaned_metrics[key] for key in ('macro_f1', 'accuracy', 'multiclass_log_loss', 'recyclable_to_organic', 'organic_to_recyclable', 'total_errors')}})
    write_csv(pd.DataFrame(comparison_rows), ARTIFACT_DIR / 'confirmation_fivefold_oof_comparison.csv')
    report_metadata = {**provenance(MANIFEST_HASH), 'stage': 'confirmation_fivefold_oof'}
    write_csv(report_frame(control_metrics['classification_report'], {**report_metadata, 'candidate': 'C0'}), ARTIFACT_DIR / 'confirmation_control_classification_report.csv')
    write_csv(report_frame(candidate_metrics['classification_report'], {**report_metadata, 'candidate': SELECTED_CANDIDATE}), ARTIFACT_DIR / 'confirmation_selected_classification_report.csv')
    fold_frame = pd.DataFrame(per_fold_rows)
    positive_net = fold_frame['net_corrections'].clip(lower=0)
    max_fold_share = float(positive_net.max() / positive_net.sum()) if positive_net.sum() > 0 else 1.0
    ro_errors = candidate_metrics['recyclable_to_organic'] + candidate_metrics['organic_to_recyclable']
    conditions = {'macro_f1': candidate_metrics['macro_f1'] >= FULL_GATES['minimum_macro_f1'], 'gain': candidate_metrics['macro_f1'] - control_metrics['macro_f1'] >= FULL_GATES['minimum_gain'], 'folds_beating': int(fold_frame['candidate_beats_control'].sum()) >= FULL_GATES['minimum_folds_beating_control'], 'total_errors_cap': candidate_metrics['total_errors'] <= FULL_GATES['maximum_total_errors'], 'ro_errors': ro_errors < FULL_GATES['maximum_ro_errors_exclusive'], 'electronic_preservation': candidate_metrics['classification_report']['Electronic']['f1-score'] >= control_metrics['classification_report']['Electronic']['f1-score'] - FULL_GATES['maximum_electronic_f1_degradation'], 'total_errors_decline': candidate_metrics['total_errors'] < control_metrics['total_errors'], 'not_one_fold_dominated': max_fold_share <= FULL_GATES['maximum_positive_net_fold_share'], 'not_reviewed_rows_only': aggregate_paired['corrected_nonreviewed'] >= FULL_GATES['minimum_corrected_nonreviewed'], 'treatment_defensible': REVIEW_MANIFEST_VALIDATION['pass'], 'useful_diversity': aggregate_paired['disagreements'] > 0 and aggregate_paired['corrected'] >= 5, 'control_reproduced_exactly': control_exact, 'manifest_hash_frozen': MANIFEST_HASH == LOCKED_V9_CONTRACT['cleaning_manifest_sha256']}
    FULL_GATE_RESULTS = {**provenance(MANIFEST_HASH), 'pass': all(conditions.values()), 'conditions': conditions, 'observed': {'candidate_macro_f1': candidate_metrics['macro_f1'], 'control_macro_f1': control_metrics['macro_f1'], 'gain': candidate_metrics['macro_f1'] - control_metrics['macro_f1'], 'folds_beating_control': int(fold_frame['candidate_beats_control'].sum()), 'candidate_total_errors': candidate_metrics['total_errors'], 'control_total_errors': control_metrics['total_errors'], 'candidate_ro_errors': ro_errors, 'candidate_electronic_f1': candidate_metrics['classification_report']['Electronic']['f1-score'], 'control_electronic_f1': control_metrics['classification_report']['Electronic']['f1-score'], 'maximum_positive_net_fold_share': max_fold_share, **aggregate_paired}, 'gates': FULL_GATES, 'historical_folds_reused_for_confirmation': True}
    CONFIRMATION_RESULTS = {**provenance(MANIFEST_HASH), 'control_metrics': control_metrics, 'candidate_metrics': candidate_metrics, 'cleaned_component_metrics': cleaned_metrics, 'per_fold': per_fold_rows, 'paired': aggregate_paired}
    write_json(FULL_GATE_RESULTS, ARTIFACT_DIR / 'full_gate_results.json')
    write_json(CONFIRMATION_RESULTS, ARTIFACT_DIR / 'confirmation_metrics.json')
    print('Five-fold confirmation pass:', FULL_GATE_RESULTS['pass'])


## 15. Final submission-2 decision

A second-submission recipe exists only if every five-fold gate passes. V7 remains the safe first candidate. The third upload is reserved; no leaderboard mapping or submission is produced here.

In [ ]:
FINAL_ADOPTION_DECISION = None
SUBMISSION2_RECIPE = None
if RUN_MODE == 'FULL_CONFIRMATION':
    adopted = FULL_GATE_RESULTS['pass']
    FINAL_ADOPTION_DECISION = {**provenance(MANIFEST_HASH), 'adopt_v9_as_second_submission_candidate': adopted, 'selected_candidate': SELECTED_CANDIDATE if adopted else None, 'final_pipeline': SELECTED_CANDIDATE if adopted else 'V7 regularized_mlp_final', 'reason': 'Every locked five-fold adoption gate passed.' if adopted else 'At least one locked five-fold gate failed; retain V7 and do not spend an upload on V9.', 'noise_interpretation': 'Gain survived fold spread, non-reviewed correction, total-error, and class-preservation guards.' if adopted else 'Observed changes did not clear every pre-registered robustness guard and are treated as fold/manual-review noise.', 'fold0_decision': FOLD0_DECISION, 'full_gate': FULL_GATE_RESULTS, 'submission_strategy': {'submission_1': 'V7 original-data five-fold average', 'submission_2': ('V9 {}'.format(SELECTED_CANDIDATE) if adopted else 'none from V9'), 'submission_3': 'reserve for a clearly non-duplicate validated refinement'}, 'oof_score_not_mapped_to_leaderboard': True}
    SUBMISSION2_RECIPE = {**provenance(MANIFEST_HASH), 'status': 'accepted' if adopted else 'rejected', 'candidate': SELECTED_CANDIDATE if adopted else None, 'standalone_or_blend': ('fixed_50_50_C0_C2' if SELECTED_CANDIDATE == 'C3' else 'treated_standalone') if adopted else None, 'fivefold_metrics': CONFIRMATION_RESULTS['candidate_metrics'], 'control_metrics': CONFIRMATION_RESULTS['control_metrics'], 'upload_budget_rule': 'Do not consume upload 2 unless adopted; reserve upload 3 for clear OOF diversity.', 'creates_submission': False}
    write_json(FINAL_ADOPTION_DECISION, ARTIFACT_DIR / 'final_adoption_decision.json')
    write_json(SUBMISSION2_RECIPE, ARTIFACT_DIR / 'submission2_candidate_recipe.json')


## 16. End-to-end integration contract

The contract records both dropped indices and training-only relabel mappings so the eventual end-to-end notebook can reproduce the selected recipe while preserving original validation/test handling.


In [ ]:
END_TO_END_CONTRACT = None
if RUN_MODE == 'FULL_CONFIRMATION':
    adopted = FINAL_ADOPTION_DECISION['adopt_v9_as_second_submission_candidate']
    pipeline_candidates = {SELECTED_CANDIDATE} | ({'C0', 'C2'} if SELECTED_CANDIDATE == 'C3' else set())
    checkpoint_names = sorted(path.name for path in HEADS_DIR.glob('*.pt') if any('_{}_fold'.format(candidate) in path.name for candidate in pipeline_candidates))
    treatment_candidate = 'C2' if SELECTED_CANDIDATE == 'C3' else SELECTED_CANDIDATE
    selected_drop_indices = set(DROP_SETS.get(treatment_candidate, np.array([], dtype=np.int64)).tolist()) if adopted else set()
    selected_relabel_map = RELABEL_MAPS.get(treatment_candidate, {}) if adopted else {}
    dropped_rows = FINAL_REVIEW_MANIFEST[FINAL_REVIEW_MANIFEST['sample_index'].astype(int).isin(selected_drop_indices)].to_dict(orient='records')
    relabeled_rows = FINAL_REVIEW_MANIFEST[FINAL_REVIEW_MANIFEST['sample_index'].astype(int).isin(selected_relabel_map)].to_dict(orient='records')
    END_TO_END_CONTRACT = {**provenance(MANIFEST_HASH), 'status': 'ready_for_integration' if adopted else 'rejected_no_v9_integration', 'model_id': MODEL_ID, 'immutable_revision': MODEL_REVISION, 'processor_contract': V7_MODEL_AUDIT.get('processor'), 'image_size': 256, 'feature_extraction': 'official final LayerNorm per token; final-block concat(CLS, mean true patch tokens)', 'backbone_frozen': True, 'treatment_manifest': {'sha256': MANIFEST_HASH, 'treatment_candidate': treatment_candidate if adopted else None, 'dropped_rows': dropped_rows, 'training_only_relabeled_rows': relabeled_rows, 'training_only_relabel_map': {str(key): int(value) for key, value in sorted(selected_relabel_map.items())}, 'validation_labels_original': True}, 'fold_assignment_sha256': FOLD_SIGNATURE_EXPECTED, 'folds': list(range(N_FOLDS)), 'head_class': 'RegularizedHead', 'head': LOCKED_V9_CONTRACT['head'], 'selected_candidate': SELECTED_CANDIDATE if adopted else None, 'per_fold_checkpoint_names': checkpoint_names, 'probability_aggregation_order': 'float64 row-normalize each fold; arithmetic mean folds in index order 0..4; row-normalize again', 'fixed_blend': {'C0': BLEND_WEIGHT_C0, 'C2': BLEND_WEIGHT_C2} if adopted and SELECTED_CANDIDATE == 'C3' else None, 'output_label_mapping': CLASS_MAPPING, 'future_test_contract': 'Extract each future test image exactly once with the locked processor/backbone; never use predictions for training, calibration, cleaning, or model selection.', 'expected_filenames': {'safe_v7': 'submission_v7_original_fivefold.csv', 'second_candidate': 'submission_v9_drop_vs_relabel_fivefold.csv'}, 'pre_submission_fail_closed_checks': ['reproduce locked five-fold OOF control within 0.0002', 'verify treatment-manifest hash and sample identities', 'verify checkpoint metadata and fold coverage', 'verify finite normalized probabilities and output mapping'], 'official_test_accessed_in_v9': False, 'submission_created_in_v9': False}
    write_json(END_TO_END_CONTRACT, ARTIFACT_DIR / 'end_to_end_integration_contract.json')


## 17. Artifact inventory and concise conclusion

The manifest inventories mode-specific outputs and records whether V9 reached audit, screen, or adoption. Runtime success and semantic review quality remain Kaggle/manual responsibilities.

In [ ]:
required = ['competition_clarification_record.json', 'leaderboard_context.json', 'v7_artifact_audit.json', 'v8_artifact_audit.json', 'cache_compatibility_audit.json', 'approach_decision.json']
if RUN_MODE == 'AUDIT':
    required += ['cleaning_candidate_ranking.csv', 'cleaning_review_manifest.csv', 'cleaning_candidate_summary.json', 'cleaning_candidate_confidence_distributions.png', 'audit_mode_decision.json'] + CONTACT_SHEET_FILES
elif RUN_MODE == 'SCREEN':
    required += ['cleaning_candidate_ranking.csv', 'cleaning_candidate_summary.json', 'cleaning_review_manifest.csv', 'locked_v9_contract.json', 'review_manifest_validation.json', 'cleaning_manifest_final.csv', 'cleaning_manifest_sha256.txt', 'control_reproduction.json', 'fold0_candidate_comparison.csv', 'fold0_candidate_metrics.json', 'fold0_paired_error_delta.csv', 'fold0_corrected_samples.csv', 'fold0_regressed_samples.csv', 'fold0_treatment_impact.csv', 'fold0_confusion_comparison.png', 'fold0_confidence_margin_distributions.png', 'fold0_diagnostic_summary.json', 'fold0_decision.json']
    for candidate in ['C0'] + LOCKED_V9_CONTRACT['enabled_candidates']:
        stem = 'heads/screen_{}_fold0_seed{}'.format(candidate, SEED)
        required.extend(stem + suffix for suffix in ('.pt', '_metrics.json', '_history.csv', '_classification_report.csv', '_predictions.csv'))
    required += REVIEW_VISUAL_FILES
else:
    required += ['cleaning_candidate_ranking.csv', 'cleaning_candidate_summary.json', 'cleaning_review_manifest.csv', 'locked_v9_contract.json', 'review_manifest_validation.json', 'cleaning_manifest_final.csv', 'cleaning_manifest_sha256.txt', 'control_reproduction.json', 'fold0_candidate_comparison.csv', 'fold0_candidate_metrics.json', 'fold0_paired_error_delta.csv', 'fold0_corrected_samples.csv', 'fold0_regressed_samples.csv', 'fold0_treatment_impact.csv', 'fold0_confusion_comparison.png', 'fold0_confidence_margin_distributions.png', 'fold0_diagnostic_summary.json', 'fold0_decision.json', 'confirmation_per_fold_comparison.csv', 'confirmation_fivefold_oof_comparison.csv', 'confirmation_control_oof_predictions.csv', 'confirmation_selected_oof_predictions.csv', 'confirmation_control_classification_report.csv', 'confirmation_selected_classification_report.csv', 'confirmation_paired_error_delta.csv', 'confirmation_corrected_samples.csv', 'confirmation_regressed_samples.csv', 'confirmation_treatment_impact.csv', 'confirmation_confusion_comparison.png', 'confirmation_confidence_margin_distributions.png', 'confirmation_diagnostic_summary.json', 'confirmation_metrics.json', 'full_gate_results.json', 'final_adoption_decision.json', 'submission2_candidate_recipe.json', 'end_to_end_integration_contract.json']
    if SELECTED_CANDIDATE == 'C3':
        required.append('confirmation_cleaned_oof_predictions.csv')
    fold0_head_candidates = ['C0', SELECTED_CANDIDATE] + (['C2'] if SELECTED_CANDIDATE == 'C3' else [])
    for candidate in sorted(set(fold0_head_candidates)):
        stem = 'heads/screen_{}_fold0_seed{}'.format(candidate, SEED)
        required.extend(stem + suffix for suffix in ('.pt', '_metrics.json', '_history.csv', '_classification_report.csv', '_predictions.csv'))
    confirmation_head_candidates = ['C0', SELECTED_CANDIDATE] + (['C2'] if SELECTED_CANDIDATE == 'C3' else [])
    for fold in CONFIRMATION_FOLDS:
        for candidate in sorted(set(confirmation_head_candidates)):
            stem = 'heads/confirmation_{}_fold{}_seed{}'.format(candidate, fold, SEED)
            required.extend(stem + suffix for suffix in ('.pt', '_metrics.json', '_history.csv', '_classification_report.csv', '_predictions.csv'))
    required += REVIEW_VISUAL_FILES
missing = [name for name in required if not (ARTIFACT_DIR / name).is_file()]
require(not missing, 'Required mode artifacts missing: {}.'.format(missing))
drop_records = FINAL_REVIEW_MANIFEST[FINAL_REVIEW_MANIFEST['review_action'].isin(('DROP_OBVIOUS_MISLABEL', 'DROP_CORRUPT_OR_UNUSABLE'))].to_dict(orient='records') if RUN_MODE != 'AUDIT' else []
relabel_records = FINAL_REVIEW_MANIFEST[FINAL_REVIEW_MANIFEST['review_action'] == 'RELABEL_OBVIOUS_MISLABEL'].to_dict(orient='records') if RUN_MODE != 'AUDIT' else []
conclusion = ('AUDIT produced a bounded OOF/objective review bundle; no model trained.' if RUN_MODE == 'AUDIT' else (FOLD0_DECISION['reason'] if RUN_MODE == 'SCREEN' else FINAL_ADOPTION_DECISION['reason']))
RUN_MANIFEST = {**provenance(MANIFEST_HASH), 'experiment_name': EXPERIMENT_NAME, 'approach': APPROACH_DECISION, 'competition_clarifications': COMPETITION_CLARIFICATIONS, 'leaderboard_context': LEADERBOARD_CONTEXT, 'v7_root': str(V7_ROOT), 'v8_root': str(V8_ROOT), 'run_mode': RUN_MODE, 'head_architecture': 'LayerNorm(8192)-Linear(512)-GELU-Dropout(0.40)-Linear(3)', 'hyperparameters': {'seed': SEED, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'batch_size': TRAIN_BATCH_SIZE, 'label_smoothing': LABEL_SMOOTHING, 'gradient_clip': GRAD_CLIP}, 'gates': {'fold0': FOLD0_GATES, 'full': FULL_GATES}, 'deleted_samples': drop_records, 'deleted_sample_indices': [int(row['sample_index']) for row in drop_records], 'relabeled_samples': relabel_records, 'training_only_relabel_map': {str(int(row['sample_index'])): int(row['reviewed_label']) for row in relabel_records}, 'conclusion': conclusion, 'fold0_decision': FOLD0_DECISION, 'full_adoption_decision': FINAL_ADOPTION_DECISION, 'competition_compliance': {'official_train_only': True, 'visual_information_only': True, 'external_data': False, 'backbone_loaded_or_forwarded': False, 'backbone_frozen': True, 'validation_filtered': False, 'training_labels_modified_for_locked_rows': bool(relabel_records), 'validation_labels_modified': False, 'test_inspected_enumerated_hashed_decoded_embedded_or_inferred': False, 'test_driven_training_tuning_calibration_or_cleaning': False, 'template_accessed': False, 'pseudo_labels': False, 'submission_created': False}, 'limitations': ['Semantic drop/relabel decisions depend on documented human judgment.', 'R/O packaging versus contents remains unresolved and defaults to KEEP.', 'Historical folds are reused for confirmation, not pristine assessment.', 'Static notebook creation does not establish Kaggle runtime success.'], 'elapsed_seconds': time.perf_counter() - STARTED_AT}
write_json(RUN_MANIFEST, ARTIFACT_DIR / 'run_manifest.json')
inventory = sorted(str(path.relative_to(ARTIFACT_DIR)) for path in ARTIFACT_DIR.rglob('*') if path.is_file())
RUN_MANIFEST['artifact_inventory'] = inventory
RUN_MANIFEST['artifact_count'] = len(inventory)
write_json(RUN_MANIFEST, ARTIFACT_DIR / 'run_manifest.json')
print(conclusion)
print('Artifacts:', len(inventory), '| official test accessed: False | submission created: False')
